# Fold 1 — actual training

Extracted from the original HISS(6).ipynb, cell 257. Code is preserved for audit; paths and dependencies are documented in README.


# Fold 2 — training

Extracted from the original HISS(6).ipynb, cell 300. Code is preserved for audit; paths and dependencies are documented in README.


# Fold 3 — corrected training

Extracted from the original HISS(6).ipynb, cell 315. Code is preserved for audit; paths and dependencies are documented in README.


# Fold 4 — fresh self-contained training

Extracted from the original HISS(6).ipynb, cell 352. Code is preserved for audit; paths and dependencies are documented in README.


# Fold 5 — actual model training

Extracted from the original HISS(6).ipynb, cell 363. Code is preserved for audit; paths and dependencies are documented in README.


In [ ]:
# ============================================================
# NEUROTRUSTFORMER — FOLD 1 ACTUAL GPU TRAINING
# ============================================================

import os
import gc
import json
import time
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import mne

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")

# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/NeuroTrustFormer_HISS"
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
MODELS_DIR = os.path.join(PROJECT_DIR, "models")
CODE_DIR = os.path.join(PROJECT_DIR, "code")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

metadata_file = os.path.join(
    RESULTS_DIR,
    "CHBMIT_FOLD1_verified_training_metadata_canonical_v2.csv"
)

split_file = os.path.join(
    RESULTS_DIR,
    "CHBMIT_locked_case_level_5fold_split.csv"
)

normalizer_file = os.path.join(
    RESULTS_DIR,
    "CHBMIT_FOLD1_training_normalization_parameters.npz"
)

model_file = os.path.join(
    CODE_DIR,
    "neurotrustformer_eeg_model.py"
)

best_checkpoint = os.path.join(
    MODELS_DIR,
    "NeuroTrustFormer_CHBMIT_FOLD1_best.pt"
)

last_checkpoint = os.path.join(
    MODELS_DIR,
    "NeuroTrustFormer_CHBMIT_FOLD1_last.pt"
)

history_file = os.path.join(
    RESULTS_DIR,
    "CHBMIT_FOLD1_training_history.csv"
)

runtime_config_file = os.path.join(
    RESULTS_DIR,
    "CHBMIT_FOLD1_actual_training_config.json"
)

# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

SEED = 2026

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

print("=" * 70)
print("NEUROTRUSTFORMER — FOLD 1 GPU TRAINING")
print("=" * 70)

# ============================================================
# 3. DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nDevice:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024**3),
            2
        ),
        "GB"
    )
else:
    raise RuntimeError(
        "CUDA GPU is not available. "
        "Do not continue training on CPU."
    )

# ============================================================
# 4. CONFIGURATION
# ============================================================

BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 5
GRADIENT_CLIP = 1.0

N_CHANNELS = 17
N_SAMPLES = 2560
D_MODEL = 128
N_HEADS = 4
N_LAYERS = 2
DROPOUT = 0.2

# ============================================================
# 5. LOAD DATA
# ============================================================

metadata = pd.read_csv(
    metadata_file,
    low_memory=False
)

split_df = pd.read_csv(split_file)

# ------------------------------------------------------------
# Locked Fold-1 cases
# ------------------------------------------------------------

fold1 = split_df[
    split_df["fold"] == 1
].copy()

train_cases = sorted(
    fold1.loc[
        fold1["role"] == "train",
        "case"
    ].astype(str).unique()
)

val_cases = sorted(
    fold1.loc[
        fold1["role"] == "validation",
        "case"
    ].astype(str).unique()
)

test_cases = sorted(
    fold1.loc[
        fold1["role"] == "test",
        "case"
    ].astype(str).unique()
)

print("\nLocked Fold-1 split:")
print("Training   :", train_cases)
print("Validation :", val_cases)
print("Test       :", test_cases)

# ============================================================
# IMPORTANT:
# The verified training metadata contains the 15 training
# cases only. Validation metadata is constructed from any
# available verified metadata files in the project.
# ============================================================

# ------------------------------------------------------------
# Search all CSV metadata files for validation cases
# ------------------------------------------------------------

csv_candidates = []

for root, dirs, files in os.walk(RESULTS_DIR):

    for filename in files:

        if not filename.lower().endswith(".csv"):
            continue

        path = os.path.join(root, filename)

        try:
            temp = pd.read_csv(
                path,
                low_memory=False,
                nrows=5
            )

            required = {
                "case",
                "recording",
                "start_sec",
                "label"
            }

            if required.issubset(set(temp.columns)):
                csv_candidates.append(path)

        except Exception:
            pass

print(
    "\nMetadata candidates found:",
    len(csv_candidates)
)

# ------------------------------------------------------------
# Collect validation rows
# ------------------------------------------------------------

validation_parts = []

for path in csv_candidates:

    try:

        temp = pd.read_csv(
            path,
            low_memory=False
        )

        if not {"case", "recording", "start_sec", "label"}.issubset(
            set(temp.columns)
        ):
            continue

        temp["case"] = temp["case"].astype(str)

        part = temp[
            temp["case"].isin(val_cases)
        ].copy()

        if len(part) > 0:
            validation_parts.append(part)

    except Exception:
        continue

if len(validation_parts) == 0:

    raise RuntimeError(
        "No validation metadata was found for the four "
        "locked Fold-1 validation cases. "
        "Training has NOT started."
    )

validation_metadata = pd.concat(
    validation_parts,
    ignore_index=True
)

# ------------------------------------------------------------
# Remove duplicate physical windows
# ------------------------------------------------------------

validation_metadata = validation_metadata.drop_duplicates(
    subset=[
        "case",
        "recording",
        "window_index"
    ],
    keep="first"
).reset_index(drop=True)

# ------------------------------------------------------------
# Verify validation cases
# ------------------------------------------------------------

found_val_cases = set(
    validation_metadata["case"].astype(str).unique()
)

missing_val_cases = set(val_cases) - found_val_cases

if missing_val_cases:
    raise RuntimeError(
        f"Validation metadata is missing cases: "
        f"{sorted(missing_val_cases)}. "
        f"Training has NOT started."
    )

# ------------------------------------------------------------
# Training metadata
# ------------------------------------------------------------

training_metadata = metadata.copy()

training_metadata["case"] = (
    training_metadata["case"].astype(str)
)

training_metadata = training_metadata[
    training_metadata["case"].isin(train_cases)
].copy()

# ============================================================
# 6. VERIFY NO LEAKAGE
# ============================================================

assert set(training_metadata["case"]).isdisjoint(
    set(validation_metadata["case"])
)

assert set(training_metadata["case"]).isdisjoint(
    set(test_cases)
)

assert set(validation_metadata["case"]).isdisjoint(
    set(test_cases)
)

print("\n" + "=" * 70)
print("DATASET SIZES")
print("=" * 70)

print(
    f"Training windows   : {len(training_metadata):,}"
)

print(
    f"Validation windows : {len(validation_metadata):,}"
)

print(
    f"Training cases     : "
    f"{training_metadata['case'].nunique()}"
)

print(
    f"Validation cases   : "
    f"{validation_metadata['case'].nunique()}"
)

print(
    f"Test cases         : "
    f"{len(test_cases)} "
    "(UNTOUCHED)"
)

# ============================================================
# 7. CLASS WEIGHTS
# ============================================================

train_counts = (
    training_metadata["label"]
    .astype(int)
    .value_counts()
    .sort_index()
)

n_negative = int(train_counts.get(0, 0))
n_positive = int(train_counts.get(1, 0))
n_total = n_negative + n_positive

class_weights = np.array(
    [
        n_total / (2.0 * n_negative),
        n_total / (2.0 * n_positive)
    ],
    dtype=np.float32
)

print("\n" + "=" * 70)
print("CLASS BALANCE")
print("=" * 70)

print("Negative:", f"{n_negative:,}")
print("Positive:", f"{n_positive:,}")
print(
    "Ratio:",
    f"{n_negative / n_positive:.2f}:1"
)

print(
    "Class weights:",
    class_weights
)

# ============================================================
# 8. NORMALIZATION
# ============================================================

norm = np.load(normalizer_file)

means = norm["mean"].astype(np.float32)
stds = norm["std"].astype(np.float32)

assert means.shape == (17,)
assert stds.shape == (17,)

# ============================================================
# 9. CHANNEL DEFINITIONS
# ============================================================

TARGET_CHANNELS = [
    "C3-P3",
    "C4-P4",
    "CZ-PZ",
    "F3-C3",
    "F4-C4",
    "F7-T7",
    "F8-T8",
    "FP1-F3",
    "FP1-F7",
    "FP2-F4",
    "FP2-F8",
    "FZ-CZ",
    "P3-O1",
    "P4-O2",
    "P7-O1",
    "P8-O2",
    "T7-P7",
]

TARGET_CHANNELS = [
    x.upper() for x in TARGET_CHANNELS
]


def normalize_channel_name(name):

    name = str(name).upper().strip()
    name = name.replace(" ", "")
    name = name.replace("_", "-")

    return name


# ============================================================
# 10. EDF LOCATOR
# ============================================================

def locate_edf(recording, record_path=None):

    candidates = []

    if isinstance(record_path, str):

        if record_path.lower().endswith(".edf"):
            candidates.append(record_path)

    candidates.extend([
        os.path.join(
            "/content/chbmit_raw",
            recording
        ),
        os.path.join(
            PROJECT_DIR,
            "data",
            "chbmit",
            recording
        ),
        os.path.join(
            PROJECT_DIR,
            "data",
            "CHB-MIT",
            recording
        ),
        os.path.join(
            PROJECT_DIR,
            "data",
            "chbmit_raw",
            recording
        ),
    ])

    for path in candidates:

        if os.path.exists(path):
            return path

    # Recursive search
    for root in [
        "/content/chbmit_raw",
        os.path.join(PROJECT_DIR, "data")
    ]:

        if not os.path.exists(root):
            continue

        for dirpath, _, filenames in os.walk(root):

            if recording in filenames:

                return os.path.join(
                    dirpath,
                    recording
                )

    return None


# ============================================================
# 11. RECORDING CACHE
# ============================================================

class RecordingCache:

    def __init__(self):

        self.recording = None
        self.data = None
        self.availability = None

    def clear(self):

        self.recording = None
        self.data = None
        self.availability = None

        gc.collect()

    def load(
        self,
        recording,
        record_path=None
    ):

        if self.recording == recording:
            return (
                self.data,
                self.availability
            )

        self.clear()

        edf_path = locate_edf(
            recording,
            record_path
        )

        if edf_path is None:

            raise FileNotFoundError(
                f"EDF not found: {recording}"
            )

        raw = mne.io.read_raw_edf(
            edf_path,
            preload=False,
            verbose="ERROR"
        )

        sfreq = float(raw.info["sfreq"])

        if abs(sfreq - 256.0) > 1e-6:

            raise ValueError(
                f"{recording}: expected 256 Hz, "
                f"found {sfreq}"
            )

        channel_map = {}

        for idx, channel_name in enumerate(
            raw.ch_names
        ):

            normalized = normalize_channel_name(
                channel_name
            )

            if normalized in TARGET_CHANNELS:

                if normalized in channel_map:

                    channel_map[normalized] = None

                else:

                    channel_map[normalized] = idx

        data = np.zeros(
            (17, raw.n_times),
            dtype=np.float32
        )

        availability = np.zeros(
            17,
            dtype=np.float32
        )

        for target_idx, target_name in enumerate(
            TARGET_CHANNELS
        ):

            source_idx = channel_map.get(
                target_name
            )

            if source_idx is None:
                continue

            channel_data = raw.get_data(
                picks=[source_idx]
            )[0]

            data[target_idx] = (
                channel_data.astype(
                    np.float32,
                    copy=False
                )
            )

            availability[target_idx] = 1.0

        del raw

        self.recording = recording
        self.data = data
        self.availability = availability

        return data, availability


cache = RecordingCache()


# ============================================================
# 12. BATCH GENERATOR
# ============================================================

def generate_batches(
    dataframe,
    batch_size,
    shuffle_recordings=True,
    shuffle_windows=True
):

    df = dataframe.copy()

    recordings = (
        df["recording"]
        .astype(str)
        .drop_duplicates()
        .tolist()
    )

    if shuffle_recordings:

        random.shuffle(recordings)

    for recording in recordings:

        rows = df[
            df["recording"].astype(str)
            == recording
        ].copy()

        if shuffle_windows:

            row_indices = np.arange(
                len(rows)
            )

            np.random.shuffle(
                row_indices
            )

            rows = rows.iloc[
                row_indices
            ].reset_index(drop=True)

        else:

            rows = rows.sort_values(
                "window_index"
            ).reset_index(drop=True)

        record_path = rows.iloc[0][
            "record_path"
        ]

        recording_data, availability = (
            cache.load(
                recording,
                record_path
            )
        )

        for start in range(
            0,
            len(rows),
            batch_size
        ):

            batch_rows = rows.iloc[
                start:start + batch_size
            ]

            actual_batch = len(
                batch_rows
            )

            eeg_batch = np.zeros(
                (
                    actual_batch,
                    17,
                    2560
                ),
                dtype=np.float32
            )

            mask_batch = np.zeros(
                (
                    actual_batch,
                    17
                ),
                dtype=np.float32
            )

            labels = np.zeros(
                actual_batch,
                dtype=np.int64
            )

            for i, (_, row) in enumerate(
                batch_rows.iterrows()
            ):

                start_sample = int(
                    round(
                        float(row["start_sec"])
                        * 256
                    )
                )

                end_sample = (
                    start_sample + 2560
                )

                window = recording_data[
                    :,
                    start_sample:end_sample
                ]

                if window.shape != (
                    17,
                    2560
                ):

                    raise ValueError(
                        f"Invalid window shape "
                        f"{window.shape} in "
                        f"{recording}"
                    )

                normalized = window.copy()

                available = (
                    availability.astype(bool)
                )

                normalized[available] = (
                    normalized[available]
                    - means[
                        available,
                        None
                    ]
                ) / stds[
                    available,
                    None
                ]

                normalized[
                    ~available
                ] = 0.0

                eeg_batch[i] = normalized

                mask_batch[i] = availability

                labels[i] = int(
                    row["label"]
                )

            yield (
                torch.from_numpy(eeg_batch),
                torch.from_numpy(mask_batch),
                torch.from_numpy(labels)
            )


# ============================================================
# 13. LOAD MODEL
# ============================================================

import sys

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

from neurotrustformer_eeg_model import (
    NeuroTrustFormerEEG
)

model = NeuroTrustFormerEEG(
    n_channels=N_CHANNELS,
    n_samples=N_SAMPLES,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    dropout=DROPOUT
).to(device)

print("\n" + "=" * 70)
print("MODEL")
print("=" * 70)

parameter_count = sum(
    p.numel()
    for p in model.parameters()
)

print(
    "Parameters:",
    f"{parameter_count:,}"
)

# ============================================================
# 14. LOSS / OPTIMIZER
# ============================================================

weight_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)

criterion = nn.CrossEntropyLoss(
    weight=weight_tensor
)

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

# ============================================================
# 15. MIXED PRECISION
# ============================================================

scaler = torch.amp.GradScaler(
    "cuda"
)

# ============================================================
# 16. TRAINING CONFIG
# ============================================================

actual_config = {
    "seed": SEED,
    "fold": 1,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "early_stopping_patience": PATIENCE,
    "gradient_clip_norm": GRADIENT_CLIP,
    "mixed_precision": "float16",
    "device": torch.cuda.get_device_name(0),
    "loader": "recording_streaming_cache",
    "n_channels": N_CHANNELS,
    "n_samples": N_SAMPLES,
    "d_model": D_MODEL,
    "n_heads": N_HEADS,
    "n_layers": N_LAYERS,
    "dropout": DROPOUT,
    "train_cases": train_cases,
    "validation_cases": val_cases,
    "test_cases": test_cases,
    "test_used_during_training": False,
    "class_weights": class_weights.tolist(),
}

with open(
    runtime_config_file,
    "w"
) as f:

    json.dump(
        actual_config,
        f,
        indent=2
    )

# ============================================================
# 17. METRICS
# ============================================================

def calculate_metrics(
    y_true,
    y_probability,
    average_loss
):

    y_true = np.asarray(
        y_true,
        dtype=np.int64
    )

    y_probability = np.asarray(
        y_probability,
        dtype=np.float64
    )

    y_pred = (
        y_probability >= 0.5
    ).astype(np.int64)

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    try:

        auc = roc_auc_score(
            y_true,
            y_probability
        )

    except Exception:

        auc = float("nan")

    try:

        ap = average_precision_score(
            y_true,
            y_probability
        )

    except Exception:

        ap = float("nan")

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    return {
        "loss": float(average_loss),
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "f1": float(f1),
        "roc_auc": float(auc),
        "pr_auc": float(ap),
        "tp": int(tp),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
    }


# ============================================================
# 18. VALIDATION
# ============================================================

def validate():

    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_labels = []
    all_probabilities = []

    with torch.no_grad():

        for eeg, mask, labels in generate_batches(
            validation_metadata,
            BATCH_SIZE,
            shuffle_recordings=False,
            shuffle_windows=False
        ):

            eeg = eeg.to(
                device,
                non_blocking=True
            )

            mask = mask.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                output = model(
                    eeg,
                    channel_mask=mask
                )

                logits = output["logits"]

                loss = criterion(
                    logits,
                    labels
                )

            batch_size_actual = (
                labels.size(0)
            )

            total_loss += (
                loss.item()
                * batch_size_actual
            )

            total_samples += (
                batch_size_actual
            )

            probabilities = torch.softmax(
                logits.float(),
                dim=1
            )[:, 1]

            all_labels.extend(
                labels.detach()
                .cpu()
                .numpy()
                .tolist()
            )

            all_probabilities.extend(
                probabilities.detach()
                .cpu()
                .numpy()
                .tolist()
            )

    average_loss = (
        total_loss / total_samples
    )

    return calculate_metrics(
        all_labels,
        all_probabilities,
        average_loss
    )


# ============================================================
# 19. RESUME CHECKPOINT
# ============================================================

start_epoch = 1
best_val_loss = float("inf")
patience_counter = 0
history = []

if os.path.exists(last_checkpoint):

    print("\n" + "=" * 70)
    print("RESUMING FROM EXISTING CHECKPOINT")
    print("=" * 70)

    checkpoint = torch.load(
        last_checkpoint,
        map_location=device,
        weights_only=False
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    scheduler.load_state_dict(
        checkpoint["scheduler_state_dict"]
    )

    if checkpoint.get(
        "scaler_state_dict"
    ) is not None:

        scaler.load_state_dict(
            checkpoint["scaler_state_dict"]
        )

    start_epoch = (
        int(checkpoint["epoch"]) + 1
    )

    best_val_loss = float(
        checkpoint["best_val_loss"]
    )

    patience_counter = int(
        checkpoint["patience_counter"]
    )

    if os.path.exists(history_file):

        history = pd.read_csv(
            history_file
        ).to_dict("records")

    print(
        "Resuming at epoch:",
        start_epoch
    )

else:

    print("\nNo previous checkpoint found.")
    print("Starting training from epoch 1.")

# ============================================================
# 20. TRAINING LOOP
# ============================================================

for epoch in range(
    start_epoch,
    EPOCHS + 1
):

    epoch_start = time.time()

    model.train()

    running_loss = 0.0
    running_samples = 0

    train_true = []
    train_prob = []

    print("\n" + "=" * 70)
    print(
        f"EPOCH {epoch}/{EPOCHS}"
    )
    print("=" * 70)

    # --------------------------------------------------------
    # Training batches
    # --------------------------------------------------------

    for batch_idx, (
        eeg,
        mask,
        labels
    ) in enumerate(
        generate_batches(
            training_metadata,
            BATCH_SIZE,
            shuffle_recordings=True,
            shuffle_windows=True
        ),
        start=1
    ):

        eeg = eeg.to(
            device,
            non_blocking=True
        )

        mask = mask.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            output = model(
                eeg,
                channel_mask=mask
            )

            logits = output["logits"]

            loss = criterion(
                logits,
                labels
            )

        scaler.scale(
            loss
        ).backward()

        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRADIENT_CLIP
        )

        scaler.step(
            optimizer
        )

        scaler.update()

        batch_n = labels.size(0)

        running_loss += (
            loss.item() * batch_n
        )

        running_samples += batch_n

        probabilities = torch.softmax(
            logits.detach().float(),
            dim=1
        )[:, 1]

        train_true.extend(
            labels.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        train_prob.extend(
            probabilities.cpu()
            .numpy()
            .tolist()
        )

        # Progress every 100 batches
        if (
            batch_idx == 1
            or batch_idx % 100 == 0
        ):

            elapsed = (
                time.time()
                - epoch_start
            )

            print(
                f"Batch {batch_idx:4d} | "
                f"Loss {loss.item():.4f} | "
                f"Elapsed {elapsed/60:.1f} min"
            )

    # --------------------------------------------------------
    # Training metrics
    # --------------------------------------------------------

    train_loss = (
        running_loss
        / running_samples
    )

    train_metrics = calculate_metrics(
        train_true,
        train_prob,
        train_loss
    )

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    print("\nRunning validation...")

    val_metrics = validate()

    scheduler.step(
        val_metrics["loss"]
    )

    epoch_time = (
        time.time()
        - epoch_start
    )

    current_lr = optimizer.param_groups[0][
        "lr"
    ]

    # --------------------------------------------------------
    # Epoch results
    # --------------------------------------------------------

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_precision": train_metrics["precision"],
        "train_recall": train_metrics["recall"],
        "train_specificity": train_metrics["specificity"],
        "train_f1": train_metrics["f1"],
        "train_roc_auc": train_metrics["roc_auc"],
        "train_pr_auc": train_metrics["pr_auc"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_specificity": val_metrics["specificity"],
        "val_f1": val_metrics["f1"],
        "val_roc_auc": val_metrics["roc_auc"],
        "val_pr_auc": val_metrics["pr_auc"],
        "val_tp": val_metrics["tp"],
        "val_tn": val_metrics["tn"],
        "val_fp": val_metrics["fp"],
        "val_fn": val_metrics["fn"],
        "learning_rate": current_lr,
        "epoch_minutes": epoch_time / 60.0,
    }

    history.append(row)

    history_df = pd.DataFrame(
        history
    )

    history_df.to_csv(
        history_file,
        index=False
    )

    # --------------------------------------------------------
    # Print results
    # --------------------------------------------------------

    print("\n" + "-" * 70)

    print(
        f"Epoch {epoch} completed "
        f"in {epoch_time/60:.2f} minutes"
    )

    print(
        f"Train Loss : {train_metrics['loss']:.5f}"
    )

    print(
        f"Val Loss   : {val_metrics['loss']:.5f}"
    )

    print(
        f"Val Acc    : {val_metrics['accuracy']:.5f}"
    )

    print(
        f"Val Prec   : {val_metrics['precision']:.5f}"
    )

    print(
        f"Val Recall : {val_metrics['recall']:.5f}"
    )

    print(
        f"Val Spec   : {val_metrics['specificity']:.5f}"
    )

    print(
        f"Val F1     : {val_metrics['f1']:.5f}"
    )

    print(
        f"Val ROC-AUC: {val_metrics['roc_auc']:.5f}"
    )

    print(
        f"Val PR-AUC : {val_metrics['pr_auc']:.5f}"
    )

    print(
        f"Learning rate: {current_lr:.2e}"
    )

    # --------------------------------------------------------
    # Check improvement
    # --------------------------------------------------------

    improved = (
        val_metrics["loss"]
        < best_val_loss
    )

    if improved:

        best_val_loss = (
            val_metrics["loss"]
        )

        patience_counter = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict":
                    model.state_dict(),
                "optimizer_state_dict":
                    optimizer.state_dict(),
                "scheduler_state_dict":
                    scheduler.state_dict(),
                "scaler_state_dict":
                    scaler.state_dict(),
                "best_val_loss":
                    best_val_loss,
                "patience_counter":
                    patience_counter,
                "config":
                    actual_config,
                "train_cases":
                    train_cases,
                "validation_cases":
                    val_cases,
                "test_cases":
                    test_cases,
            },
            best_checkpoint
        )

        print(
            "\n✓ NEW BEST MODEL SAVED"
        )

    else:

        patience_counter += 1

        print(
            f"\nNo improvement. "
            f"Patience: "
            f"{patience_counter}/{PATIENCE}"
        )

    # --------------------------------------------------------
    # Save latest checkpoint
    # --------------------------------------------------------

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict":
                model.state_dict(),
            "optimizer_state_dict":
                optimizer.state_dict(),
            "scheduler_state_dict":
                scheduler.state_dict(),
            "scaler_state_dict":
                scaler.state_dict(),
            "best_val_loss":
                best_val_loss,
            "patience_counter":
                patience_counter,
            "config":
                actual_config,
            "train_cases":
                train_cases,
            "validation_cases":
                val_cases,
            "test_cases":
                test_cases,
        },
        last_checkpoint
    )

    print(
        "✓ Latest checkpoint saved."
    )

    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    if patience_counter >= PATIENCE:

        print(
            "\nEarly stopping triggered."
        )

        break

# ============================================================
# 21. FINAL STATUS
# ============================================================

print("\n" + "=" * 70)
print("TRAINING RUN FINISHED")
print("=" * 70)

print(
    "Best checkpoint:",
    best_checkpoint
)

print(
    "Latest checkpoint:",
    last_checkpoint
)

print(
    "Training history:",
    history_file
)

print(
    "\nIMPORTANT:"
)

print(
    "The Fold-1 TEST CASES were NOT used "
    "for training, model selection, or early stopping."
)

print(
    "\nDo not report these validation metrics "
    "as final test performance."
)

print("=" * 70)

# Release cache
cache.clear()

gc.collect()

torch.cuda.empty_cache()

In [ ]:
# ============================================================
# NEUROTRUSTFORMER — FOLD-2 TRAINING
# ============================================================

import os
import re
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import mne

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------
SEED = 2026

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# 2. Paths
# ------------------------------------------------------------
PROJECT_DIR = Path(
    "/content/drive/MyDrive/NeuroTrustFormer_HISS"
)

CODE_DIR = PROJECT_DIR / "code"
RESULTS_DIR = PROJECT_DIR / "results"
MODEL_DIR = PROJECT_DIR / "models"
EDF_DIR = PROJECT_DIR / "data" / "chbmit"

SPLIT_FILE = (
    RESULTS_DIR /
    "CHBMIT_locked_case_level_5fold_split.csv"
)

MASTER_METADATA = (
    RESULTS_DIR /
    "CHBMIT_MASTER_24CASE_window_metadata.csv"
)

NORMALIZATION_FILE = (
    RESULTS_DIR /
    "CHBMIT_FOLD2_training_normalization_parameters.npz"
)

MODEL_FILE = CODE_DIR / "neurotrustformer_eeg_model.py"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 3. Environment verification
# ------------------------------------------------------------
print("=" * 70)
print("NEUROTRUSTFORMER — FOLD-2 TRAINING")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not available. DO NOT continue training."
    )

DEVICE = torch.device("cuda")

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3,
        2
    ),
    "GB"
)

# ------------------------------------------------------------
# 4. Required files
# ------------------------------------------------------------
required = {
    "Split": SPLIT_FILE,
    "Master metadata": MASTER_METADATA,
    "Normalization": NORMALIZATION_FILE,
    "Model code": MODEL_FILE,
    "EDF directory": EDF_DIR,
}

print("\nRequired files:")

for name, path in required.items():
    exists = path.exists()
    print(f"{name:20s}: {exists}")
    if not exists:
        raise FileNotFoundError(str(path))

# ------------------------------------------------------------
# 5. Load model
# ------------------------------------------------------------
import sys

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from neurotrustformer_eeg_model import NeuroTrustFormerEEG

# ------------------------------------------------------------
# 6. Load split
# ------------------------------------------------------------
split_df = pd.read_csv(SPLIT_FILE)

fold2 = split_df[
    split_df["fold"] == 2
].copy()

train_cases = sorted(
    fold2.loc[
        fold2["role"] == "train",
        "case"
    ].astype(str).unique()
)

val_cases = sorted(
    fold2.loc[
        fold2["role"] == "validation",
        "case"
    ].astype(str).unique()
)

test_cases = sorted(
    fold2.loc[
        fold2["role"] == "test",
        "case"
    ].astype(str).unique()
)

print("\nFold-2 split:")
print("Train:", train_cases)
print("Validation:", val_cases)
print("Test:", test_cases)

# ------------------------------------------------------------
# 7. Load metadata
# ------------------------------------------------------------
metadata = pd.read_csv(
    MASTER_METADATA,
    low_memory=False
)

print("\nMaster metadata rows:", len(metadata))

# ------------------------------------------------------------
# 8. Correct CHB case parser
# ------------------------------------------------------------
def extract_case(filename):
    match = re.match(
        r"(chb\d+)",
        str(filename).lower()
    )
    return match.group(1) if match else None

metadata["case"] = (
    metadata["recording"]
    .astype(str)
    .apply(extract_case)
)

# ------------------------------------------------------------
# 9. Select train/validation metadata
# ------------------------------------------------------------
train_df = metadata[
    metadata["case"].isin(train_cases)
].copy()

val_df = metadata[
    metadata["case"].isin(val_cases)
].copy()

# ------------------------------------------------------------
# IMPORTANT:
# No test metadata is loaded for training.
# ------------------------------------------------------------

print("\nTraining windows:", len(train_df))
print("Validation windows:", len(val_df))

print("\nTraining labels:")
print(train_df["label"].value_counts())

print("\nValidation labels:")
print(val_df["label"].value_counts())

# ------------------------------------------------------------
# 10. Verify no case leakage
# ------------------------------------------------------------
train_case_set = set(train_df["case"])
val_case_set = set(val_df["case"])
test_case_set = set(test_cases)

assert train_case_set.isdisjoint(val_case_set)
assert train_case_set.isdisjoint(test_case_set)
assert val_case_set.isdisjoint(test_case_set)

print("\n✓ No case-level overlap between train/validation/test")

# ------------------------------------------------------------
# 11. Verify training EDF coverage
# ------------------------------------------------------------
edf_files = sorted(
    EDF_DIR.glob("*.edf")
)

training_edfs = [
    f for f in edf_files
    if extract_case(f.name) in train_cases
]

validation_edfs = [
    f for f in edf_files
    if extract_case(f.name) in val_cases
]

print("\nPhysical EDF coverage:")
print("Training EDFs:", len(training_edfs))
print("Validation EDFs:", len(validation_edfs))

if len(training_edfs) != 97:
    raise RuntimeError(
        f"Expected 97 training EDFs, found {len(training_edfs)}"
    )

# ------------------------------------------------------------
# 12. Load Fold-2 normalization
# ------------------------------------------------------------
norm = np.load(
    NORMALIZATION_FILE,
    allow_pickle=True
)

means = norm["means"].astype(np.float32)
stds = norm["stds"].astype(np.float32)

print("\nNormalization:")
print("Channels:", len(means))
print("Mean shape:", means.shape)
print("Std shape:", stds.shape)

assert len(means) == 17
assert len(stds) == 17

# ------------------------------------------------------------
# 13. Target channels
# ------------------------------------------------------------
TARGET_CHANNELS = [
    "C3-P3",
    "C4-P4",
    "CZ-PZ",
    "F3-C3",
    "F4-C4",
    "F7-T7",
    "F8-T8",
    "FP1-F3",
    "FP1-F7",
    "FP2-F4",
    "FP2-F8",
    "FZ-CZ",
    "P3-O1",
    "P4-O2",
    "P7-O1",
    "P8-O2",
    "T7-P7",
]

N_CHANNELS = 17
N_SAMPLES = 2560

# ------------------------------------------------------------
# 14. Recording cache
# ------------------------------------------------------------
class RecordingCache:

    def __init__(self):
        self.cache = {}

    def get(self, path):

        path = str(path)

        if path not in self.cache:

            raw = mne.io.read_raw_edf(
                path,
                preload=True,
                verbose=False
            )

            self.cache[path] = raw

        return self.cache[path]

    def clear(self):

        for raw in self.cache.values():
            try:
                del raw
            except:
                pass

        self.cache.clear()


CACHE = RecordingCache()

# ------------------------------------------------------------
# 15. Resolve recording path
# ------------------------------------------------------------
def get_edf_path(recording):

    path = EDF_DIR / str(recording)

    if path.exists():
        return path

    raise FileNotFoundError(
        f"EDF not found: {recording}"
    )

# ------------------------------------------------------------
# 16. Load one EEG window
# ------------------------------------------------------------
def load_eeg_window(recording, start_sec):

    path = get_edf_path(recording)

    raw = CACHE.get(path)

    # Create robust channel map
    available = {}

    for idx, name in enumerate(raw.ch_names):

        key = str(name).upper().strip()

        if key not in available:
            available[key] = idx

    eeg = np.zeros(
        (N_CHANNELS, N_SAMPLES),
        dtype=np.float32
    )

    channel_mask = np.zeros(
        N_CHANNELS,
        dtype=np.float32
    )

    start_sample = int(
        round(float(start_sec) * 256)
    )

    stop_sample = start_sample + N_SAMPLES

    for ch_idx, target in enumerate(TARGET_CHANNELS):

        key = target.upper().strip()

        if key not in available:
            continue

        raw_idx = available[key]

        data = raw.get_data(
            picks=[raw_idx],
            start=start_sample,
            stop=stop_sample,
            reject_by_annotation=None
        )[0]

        if len(data) != N_SAMPLES:
            continue

        data = data.astype(np.float32)

        # Training-only normalization
        data = (
            data - means[ch_idx]
        ) / max(stds[ch_idx], 1e-8)

        data = np.nan_to_num(
            data,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

        eeg[ch_idx] = data
        channel_mask[ch_idx] = 1.0

    return eeg, channel_mask

# ------------------------------------------------------------
# 17. Dataset
# ------------------------------------------------------------
class CHBMITDataset(Dataset):

    def __init__(self, dataframe):

        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        eeg, mask = load_eeg_window(
            row["recording"],
            row["start_sec"]
        )

        label = np.float32(row["label"])

        return (
            torch.from_numpy(eeg),
            torch.from_numpy(mask),
            torch.tensor(label, dtype=torch.long)
        )

# ------------------------------------------------------------
# 18. DataLoaders
# ------------------------------------------------------------
BATCH_SIZE = 64

train_dataset = CHBMITDataset(train_df)
val_dataset = CHBMITDataset(val_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# ------------------------------------------------------------
# 19. Model
# ------------------------------------------------------------
model = NeuroTrustFormerEEG(
    n_channels=17,
    n_samples=2560,
    d_model=128,
    n_heads=4,
    n_layers=2,
    dropout=0.2
).to(DEVICE)

print(
    "\nModel parameters:",
    sum(p.numel() for p in model.parameters())
)

# ------------------------------------------------------------
# 20. Class weights
# ------------------------------------------------------------
class_counts = (
    train_df["label"]
    .value_counts()
    .sort_index()
)

n_normal = int(
    class_counts.get(0, 0)
)

n_seizure = int(
    class_counts.get(1, 0)
)

class_weights = torch.tensor(
    [
        1.0 / max(n_normal, 1),
        1.0 / max(n_seizure, 1)
    ],
    dtype=torch.float32,
    device=DEVICE
)

class_weights = (
    class_weights /
    class_weights.sum()
    * 2.0
)

print("\nClass counts:")
print("Normal :", n_normal)
print("Seizure:", n_seizure)

print("Class weights:", class_weights.detach().cpu().numpy())

# ------------------------------------------------------------
# 21. Loss / optimizer
# ------------------------------------------------------------
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

# ------------------------------------------------------------
# 22. Metrics
# ------------------------------------------------------------
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

def calculate_metrics(y_true, y_prob):

    y_pred = (
        np.asarray(y_prob) >= 0.5
    ).astype(int)

    result = {
        "accuracy": accuracy_score(
            y_true, y_pred
        ),
        "precision": precision_score(
            y_true, y_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_true, y_pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_true, y_pred,
            zero_division=0
        )
    }

    if len(np.unique(y_true)) == 2:

        result["roc_auc"] = roc_auc_score(
            y_true,
            y_prob
        )

        result["pr_auc"] = average_precision_score(
            y_true,
            y_prob
        )

    else:

        result["roc_auc"] = np.nan
        result["pr_auc"] = np.nan

    return result

# ------------------------------------------------------------
# 23. Training configuration
# ------------------------------------------------------------
EPOCHS = 30
PATIENCE = 5

BEST_MODEL = (
    MODEL_DIR /
    "NeuroTrustFormer_FOLD2_best.pt"
)

HISTORY_FILE = (
    RESULTS_DIR /
    "CHBMIT_FOLD2_training_history.csv"
)

history = []

best_val_loss = float("inf")
patience_counter = 0

# ------------------------------------------------------------
# 24. Mixed precision
# ------------------------------------------------------------
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True
)

print("\n" + "=" * 70)
print("STARTING FOLD-2 TRAINING")
print("=" * 70)

# ------------------------------------------------------------
# 25. Training loop
# ------------------------------------------------------------
for epoch in range(1, EPOCHS + 1):

    model.train()

    train_losses = []

    for eeg, mask, labels in train_loader:

        eeg = eeg.to(
            DEVICE,
            non_blocking=True
        )

        mask = mask.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            output = model(
                eeg,
                channel_mask=mask
            )

            logits = output["logits"]

            loss = criterion(
                logits,
                labels
            )

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        scaler.step(optimizer)
        scaler.update()

        train_losses.append(
            loss.item()
        )

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------
    model.eval()

    val_losses = []
    val_true = []
    val_prob = []

    with torch.no_grad():

        for eeg, mask, labels in val_loader:

            eeg = eeg.to(
                DEVICE,
                non_blocking=True
            )

            mask = mask.to(
                DEVICE,
                non_blocking=True
            )

            labels_gpu = labels.to(
                DEVICE,
                non_blocking=True
            )

            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                output = model(
                    eeg,
                    channel_mask=mask
                )

                logits = output["logits"]

                loss = criterion(
                    logits,
                    labels_gpu
                )

                probabilities = torch.softmax(
                    logits,
                    dim=1
                )[:, 1]

            val_losses.append(
                loss.item()
            )

            val_true.extend(
                labels.numpy().tolist()
            )

            val_prob.extend(
                probabilities.float()
                .cpu()
                .numpy()
                .tolist()
            )

    train_loss = float(
        np.mean(train_losses)
    )

    val_loss = float(
        np.mean(val_losses)
    )

    metrics = calculate_metrics(
        np.asarray(val_true),
        np.asarray(val_prob)
    )

    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "roc_auc": metrics["roc_auc"],
        "pr_auc": metrics["pr_auc"],
        "learning_rate": current_lr
    }

    history.append(row)

    pd.DataFrame(history).to_csv(
        HISTORY_FILE,
        index=False
    )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss {train_loss:.4f} | "
        f"Val Loss {val_loss:.4f} | "
        f"Acc {metrics['accuracy']:.4f} | "
        f"F1 {metrics['f1']:.4f} | "
        f"ROC-AUC {metrics['roc_auc']:.4f} | "
        f"PR-AUC {metrics['pr_auc']:.4f}"
    )

    # --------------------------------------------------------
    # Save best checkpoint
    # --------------------------------------------------------
    if val_loss < best_val_loss:

        best_val_loss = val_loss
        patience_counter = 0

        torch.save(
            {
                "fold": 2,
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
                "normalization_file": str(
                    NORMALIZATION_FILE
                ),
                "seed": SEED
            },
            BEST_MODEL
        )

        print(
            f"  ✓ Best checkpoint saved "
            f"(val loss={val_loss:.6f})"
        )

    else:

        patience_counter += 1

        print(
            f"  No improvement "
            f"({patience_counter}/{PATIENCE})"
        )

        if patience_counter >= PATIENCE:

            print(
                "\n✓ Early stopping triggered."
            )

            break

# ------------------------------------------------------------
# 26. Final summary
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("FOLD-2 TRAINING COMPLETE")
print("=" * 70)

print("Best validation loss:", best_val_loss)
print("Best model:", BEST_MODEL)
print("History:", HISTORY_FILE)

print("\nIMPORTANT:")
print("The Fold-2 TEST cases were NOT used during training.")
print("Test evaluation will be performed separately.")

In [ ]:
# ==============================================================
# FOLD-3: CORRECTED NEUROTRUSTFORMER EEG TRAINING
# ==============================================================

import os
import sys
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

PROJECT = "/content/drive/MyDrive/NeuroTrustFormer_HISS"
CODE_DIR = os.path.join(PROJECT, "code")

sys.path.insert(0, CODE_DIR)

# --------------------------------------------------------------
# Existing project modules
# --------------------------------------------------------------

from neurotrustformer_eeg_model import NeuroTrustFormerEEG
from chbmit_eeg_preprocessing import load_eeg_window

# --------------------------------------------------------------
# File paths
# --------------------------------------------------------------

TRAIN_METADATA = os.path.join(
    PROJECT,
    "results",
    "CHBMIT_FOLD3_training_metadata.csv"
)

MASTER_METADATA = os.path.join(
    PROJECT,
    "results",
    "CHBMIT_MASTER_24CASE_window_metadata.csv"
)

SPLIT_FILE = os.path.join(
    PROJECT,
    "results",
    "CHBMIT_locked_case_level_5fold_split.csv"
)

NORM_FILE = os.path.join(
    PROJECT,
    "results",
    "CHBMIT_FOLD3_training_normalization_parameters.npz"
)

MODEL_OUTPUT = os.path.join(
    PROJECT,
    "models",
    "NeuroTrustFormer_FOLD3_best.pt"
)

HISTORY_OUTPUT = os.path.join(
    PROJECT,
    "results",
    "CHBMIT_FOLD3_training_history.csv"
)

CONFIG_OUTPUT = os.path.join(
    PROJECT,
    "results",
    "CHBMIT_FOLD3_training_config.json"
)

# --------------------------------------------------------------
# Reproducibility
# --------------------------------------------------------------

SEED = 2026

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --------------------------------------------------------------
# Training configuration
# --------------------------------------------------------------

BATCH_SIZE = 64
EPOCHS = 30
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 5
GRADIENT_CLIP = 1.0

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("FOLD-3 NEUROTRUSTFORMER TRAINING")
print("=" * 70)

print("\nDevice:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(
                0
            ).total_memory / (1024**3),
            2
        ),
        "GB"
    )

print(
    "\nPyTorch:",
    torch.__version__
)

# --------------------------------------------------------------
# Load metadata
# --------------------------------------------------------------

train_df = pd.read_csv(
    TRAIN_METADATA
)

master_df = pd.read_csv(
    MASTER_METADATA
)

split_df = pd.read_csv(
    SPLIT_FILE
)

print(
    "\nTraining metadata rows:",
    f"{len(train_df):,}"
)

# --------------------------------------------------------------
# Fold-3 validation/test cases
# --------------------------------------------------------------

fold3 = split_df[
    split_df["fold"] == 3
]

validation_cases = sorted(
    fold3.loc[
        fold3["role"] == "validation",
        "case"
    ].unique().tolist()
)

test_cases = sorted(
    fold3.loc[
        fold3["role"] == "test",
        "case"
    ].unique().tolist()
)

print(
    "Validation cases:",
    validation_cases
)

print(
    "Test cases:",
    test_cases
)

# --------------------------------------------------------------
# Training-only normalization
# --------------------------------------------------------------

norm = np.load(
    NORM_FILE
)

means = norm["means"].astype(
    np.float32
)

stds = norm["stds"].astype(
    np.float32
)

print(
    "\nNormalization channels:",
    len(means)
)

assert len(means) == 17
assert len(stds) == 17

# --------------------------------------------------------------
# Dataset
# --------------------------------------------------------------

class Fold3EEGDataset(Dataset):

    def __init__(
        self,
        dataframe,
        means,
        stds
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.means = means
        self.stds = stds

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # ------------------------------------------------------
        # USE THE EXISTING PROJECT EEG LOADER
        # ------------------------------------------------------

        eeg, channel_mask = load_eeg_window(
            recording=row["recording"],
            start_sec=float(row["start_sec"]),
            end_sec=float(row["end_sec"])
        )

        eeg = np.asarray(
            eeg,
            dtype=np.float32
        )

        channel_mask = np.asarray(
            channel_mask,
            dtype=np.float32
        )

        # ------------------------------------------------------
        # TRAINING-ONLY NORMALIZATION
        # ------------------------------------------------------

        eeg = (
            eeg
            - self.means[:, None]
        ) / (
            self.stds[:, None]
            + 1e-8
        )

        label = int(
            row["label"]
        )

        return (
            torch.tensor(
                eeg,
                dtype=torch.float32
            ),
            torch.tensor(
                channel_mask,
                dtype=torch.float32
            ),
            torch.tensor(
                label,
                dtype=torch.long
            )
        )

# --------------------------------------------------------------
# Training dataset
# --------------------------------------------------------------

train_dataset = Fold3EEGDataset(
    train_df,
    means,
    stds
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    drop_last=False
)

print(
    "\n✓ Training dataset created."
)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Number of batches:",
    len(train_loader)
)

# --------------------------------------------------------------
# Validation dataset
# --------------------------------------------------------------

val_df = master_df[
    master_df["case"].isin(
        validation_cases
    )
].reset_index(
    drop=True
)

print(
    "\nValidation metadata rows:",
    f"{len(val_df):,}"
)

val_dataset = Fold3EEGDataset(
    val_df,
    means,
    stds
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    drop_last=False
)

print(
    "Validation batches:",
    len(val_loader)
)

# --------------------------------------------------------------
# Smoke test
# --------------------------------------------------------------

print(
    "\nRunning first-batch smoke test..."
)

batch_eeg, batch_mask, batch_labels = next(
    iter(train_loader)
)

print(
    "EEG shape:",
    batch_eeg.shape
)

print(
    "Mask shape:",
    batch_mask.shape
)

print(
    "Labels shape:",
    batch_labels.shape
)

print(
    "Label dtype:",
    batch_labels.dtype
)

print(
    "Unique labels:",
    torch.unique(
        batch_labels
    ).tolist()
)

assert batch_eeg.shape[1] == 17
assert batch_eeg.shape[2] == 2560
assert batch_labels.dtype == torch.int64

print(
    "✓ First batch verified."
)

# --------------------------------------------------------------
# Model
# --------------------------------------------------------------

model = NeuroTrustFormerEEG(
    n_channels=17,
    n_samples=2560,
    d_model=128,
    n_heads=4,
    n_layers=2,
    dropout=0.2
).to(DEVICE)

parameter_count = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    "\nTrainable parameters:",
    f"{parameter_count:,}"
)

# --------------------------------------------------------------
# Class weights
# --------------------------------------------------------------

label_counts = (
    train_df["label"]
    .value_counts()
    .sort_index()
)

n_negative = int(
    label_counts.get(0, 0)
)

n_positive = int(
    label_counts.get(1, 0)
)

print(
    "\nTraining class distribution:"
)

print(
    "Normal :",
    f"{n_negative:,}"
)

print(
    "Seizure:",
    f"{n_positive:,}"
)

class_weights = torch.tensor(
    [
        1.0,
        n_negative / max(
            n_positive,
            1
        )
    ],
    dtype=torch.float32,
    device=DEVICE
)

print(
    "\nClass weights:",
    class_weights.detach()
    .cpu()
    .numpy()
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

# --------------------------------------------------------------
# Optimizer
# --------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

# --------------------------------------------------------------
# Training
# --------------------------------------------------------------

best_val_loss = float("inf")
best_epoch = None
patience_counter = 0

history = []

print(
    "\n" + "=" * 70
)

print(
    "STARTING FOLD-3 TRAINING"
)

print(
    "=" * 70
)

for epoch in range(
    1,
    EPOCHS + 1
):

    # ==========================================================
    # TRAIN
    # ==========================================================

    model.train()

    running_loss = 0.0
    train_count = 0

    train_true = []
    train_prob = []

    for (
        eeg,
        channel_mask,
        labels
    ) in train_loader:

        eeg = eeg.to(
            DEVICE,
            non_blocking=True
        )

        channel_mask = channel_mask.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        outputs = model(
            eeg,
            channel_mask
        )

        logits = outputs["logits"]

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRADIENT_CLIP
        )

        optimizer.step()

        running_loss += (
            loss.item()
            * labels.size(0)
        )

        train_count += labels.size(0)

        probabilities = torch.softmax(
            logits,
            dim=1
        )[:, 1]

        train_true.extend(
            labels.detach()
            .cpu()
            .numpy()
        )

        train_prob.extend(
            probabilities.detach()
            .cpu()
            .numpy()
        )

    train_loss = (
        running_loss /
        train_count
    )

    train_pred = (
        np.asarray(train_prob)
        >= 0.5
    ).astype(int)

    train_accuracy = accuracy_score(
        train_true,
        train_pred
    )

    train_precision = precision_score(
        train_true,
        train_pred,
        zero_division=0
    )

    train_recall = recall_score(
        train_true,
        train_pred,
        zero_division=0
    )

    train_f1 = f1_score(
        train_true,
        train_pred,
        zero_division=0
    )

    train_auc = roc_auc_score(
        train_true,
        train_prob
    )

    train_pr_auc = average_precision_score(
        train_true,
        train_prob
    )

    # ==========================================================
    # VALIDATION
    # ==========================================================

    model.eval()

    val_loss_sum = 0.0
    val_count = 0

    val_true = []
    val_prob = []

    with torch.no_grad():

        for (
            eeg,
            channel_mask,
            labels
        ) in val_loader:

            eeg = eeg.to(
                DEVICE,
                non_blocking=True
            )

            channel_mask = channel_mask.to(
                DEVICE,
                non_blocking=True
            )

            labels = labels.to(
                DEVICE,
                non_blocking=True
            )

            outputs = model(
                eeg,
                channel_mask
            )

            logits = outputs["logits"]

            loss = criterion(
                logits,
                labels
            )

            val_loss_sum += (
                loss.item()
                * labels.size(0)
            )

            val_count += labels.size(0)

            probabilities = torch.softmax(
                logits,
                dim=1
            )[:, 1]

            val_true.extend(
                labels.cpu().numpy()
            )

            val_prob.extend(
                probabilities.cpu().numpy()
            )

    val_loss = (
        val_loss_sum /
        val_count
    )

    val_pred = (
        np.asarray(val_prob)
        >= 0.5
    ).astype(int)

    val_accuracy = accuracy_score(
        val_true,
        val_pred
    )

    val_precision = precision_score(
        val_true,
        val_pred,
        zero_division=0
    )

    val_recall = recall_score(
        val_true,
        val_pred,
        zero_division=0
    )

    val_f1 = f1_score(
        val_true,
        val_pred,
        zero_division=0
    )

    val_auc = roc_auc_score(
        val_true,
        val_prob
    )

    val_pr_auc = average_precision_score(
        val_true,
        val_prob
    )

    # ----------------------------------------------------------
    # Scheduler
    # ----------------------------------------------------------

    scheduler.step(
        val_loss
    )

    current_lr = optimizer.param_groups[
        0
    ]["lr"]

    # ----------------------------------------------------------
    # Save history
    # ----------------------------------------------------------

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "train_precision": train_precision,
            "train_recall": train_recall,
            "train_f1": train_f1,
            "train_roc_auc": train_auc,
            "train_pr_auc": train_pr_auc,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
            "val_precision": val_precision,
            "val_recall": val_recall,
            "val_f1": val_f1,
            "val_roc_auc": val_auc,
            "val_pr_auc": val_pr_auc,
            "learning_rate": current_lr
        }
    )

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss {train_loss:.6f} | "
        f"Val Loss {val_loss:.6f} | "
        f"Val Acc {val_accuracy:.4f} | "
        f"Val Precision {val_precision:.4f} | "
        f"Val Recall {val_recall:.4f} | "
        f"Val F1 {val_f1:.4f} | "
        f"Val ROC-AUC {val_auc:.4f} | "
        f"Val PR-AUC {val_pr_auc:.4f} | "
        f"LR {current_lr:.6f}"
    )

    # ==========================================================
    # BEST CHECKPOINT
    # ==========================================================

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
                "config": {
                    "seed": SEED,
                    "batch_size": BATCH_SIZE,
                    "max_epochs": EPOCHS,
                    "learning_rate": LEARNING_RATE,
                    "weight_decay": WEIGHT_DECAY,
                    "patience": PATIENCE,
                    "gradient_clip": GRADIENT_CLIP,
                    "n_channels": 17,
                    "n_samples": 2560,
                    "d_model": 128,
                    "n_heads": 4,
                    "n_layers": 2,
                    "dropout": 0.2
                }
            },
            MODEL_OUTPUT
        )

        print(
            f"  ✓ New best checkpoint saved "
            f"(epoch {epoch}, "
            f"val loss {val_loss:.6f})"
        )

    else:

        patience_counter += 1

        print(
            f"  No improvement. "
            f"Early-stop counter: "
            f"{patience_counter}/{PATIENCE}"
        )

        if patience_counter >= PATIENCE:

            print(
                f"\nEarly stopping at epoch {epoch}."
            )

            break

# --------------------------------------------------------------
# Save history
# --------------------------------------------------------------

history_df = pd.DataFrame(
    history
)

history_df.to_csv(
    HISTORY_OUTPUT,
    index=False
)

# --------------------------------------------------------------
# Save configuration
# --------------------------------------------------------------

config = {
    "fold": 3,
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "max_epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "early_stopping_patience": PATIENCE,
    "gradient_clip": GRADIENT_CLIP,
    "device": str(DEVICE),
    "model": {
        "n_channels": 17,
        "n_samples": 2560,
        "d_model": 128,
        "n_heads": 4,
        "n_layers": 2,
        "dropout": 0.2
    },
    "training_cases": sorted(
        train_df["case"]
        .unique()
        .tolist()
    ),
    "validation_cases": validation_cases,
    "test_cases": test_cases,
    "best_epoch": best_epoch,
    "best_validation_loss": best_val_loss
}

with open(
    CONFIG_OUTPUT,
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=2
    )

# --------------------------------------------------------------
# Final output
# --------------------------------------------------------------

print(
    "\n" + "=" * 70
)

print(
    "FOLD-3 TRAINING COMPLETE"
)

print(
    "=" * 70
)

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best validation loss:",
    best_val_loss
)

print(
    "\nBest checkpoint:"
)

print(
    MODEL_OUTPUT
)

print(
    "\nTraining history:"
)

print(
    HISTORY_OUTPUT
)

print(
    "\nConfiguration:"
)

print(
    CONFIG_OUTPUT
)

print(
    "\n✓ TEST CASES WERE NOT USED FOR MODEL SELECTION"
)

print(
    "✓ FOLD-3 TRAINING COMPLETE"
)

print(
    "=" * 70
)

In [ ]:
# ============================================================
# FOLD-4 NEUROTRUSTFORMER — FRESH SELF-CONTAINED TRAINING
# ============================================================
# Training:
#   15 Fold-4 training cases
#
# Validation:
#   chb14, chb17, chb20, chb22
#
# Test:
#   chb08, chb18, chb21, chb23, chb24
#   NOT TOUCHED IN THIS CELL
#
# Best checkpoint = lowest validation loss
# ============================================================

import os
import sys
import json
import random
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import mne

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

BASE = "/content/drive/MyDrive/NeuroTrustFormer_HISS"

DATA_DIR = f"{BASE}/data/chbmit"
RESULTS_DIR = f"{BASE}/results"
MODELS_DIR = f"{BASE}/models"
CODE_DIR = f"{BASE}/code"

TRAIN_META_FILE = (
    f"{RESULTS_DIR}/CHBMIT_FOLD4_training_metadata.csv"
)

MASTER_META_FILE = (
    f"{RESULTS_DIR}/CHBMIT_MASTER_24CASE_window_metadata.csv"
)

NORM_FILE = (
    f"{RESULTS_DIR}/"
    f"CHBMIT_FOLD4_training_normalization_parameters.npz"
)

CONFIG_FILE = (
    f"{RESULTS_DIR}/CHBMIT_FOLD4_training_config.json"
)

BEST_MODEL_FILE = (
    f"{MODELS_DIR}/NeuroTrustFormer_FOLD4_best.pt"
)

HISTORY_FILE = (
    f"{RESULTS_DIR}/CHBMIT_FOLD4_training_history.csv"
)

MODEL_FILE = (
    f"{CODE_DIR}/neurotrustformer_eeg_model.py"
)

# ------------------------------------------------------------
# 2. FIXED EXPERIMENT SETTINGS
# ------------------------------------------------------------

SEED = 2026
FOLD = 4

TRAIN_CASES = [
    "chb01", "chb02", "chb03", "chb04", "chb05",
    "chb06", "chb07", "chb09", "chb10", "chb11",
    "chb12", "chb13", "chb15", "chb16", "chb19"
]

VAL_CASES = [
    "chb14", "chb17", "chb20", "chb22"
]

TEST_CASES = [
    "chb08", "chb18", "chb21", "chb23", "chb24"
]

BATCH_SIZE = 64
MAX_EPOCHS = 30

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

EARLY_STOPPING_PATIENCE = 5

GRADIENT_CLIP = 1.0

SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 2

N_CHANNELS = 17
N_SAMPLES = 2560

D_MODEL = 128
N_HEADS = 4
N_LAYERS = 2
DROPOUT = 0.20

# ------------------------------------------------------------
# 3. REPRODUCIBILITY
# ------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# 4. GPU
# ------------------------------------------------------------

assert torch.cuda.is_available(), (
    "CUDA GPU is required for Fold-4 training."
)

DEVICE = torch.device("cuda")

print("=" * 70)
print("FOLD-4 NEUROTRUSTFORMER TRAINING")
print("=" * 70)

print("\nGPU:")
print(torch.cuda.get_device_name(0))

gpu_memory = (
    torch.cuda.get_device_properties(0)
    .total_memory / (1024 ** 3)
)

print(
    f"GPU memory: {gpu_memory:.2f} GB"
)

# ------------------------------------------------------------
# 5. REQUIRED FILE CHECK
# ------------------------------------------------------------

required_files = {
    "training metadata": TRAIN_META_FILE,
    "master metadata": MASTER_META_FILE,
    "normalization": NORM_FILE,
    "configuration": CONFIG_FILE,
    "model code": MODEL_FILE
}

print("\nREQUIRED FILES")
print("-" * 70)

for name, path in required_files.items():

    assert os.path.exists(path), (
        f"Missing {name}:\n{path}"
    )

    print(f"✓ {name}")

# ------------------------------------------------------------
# 6. LOAD CONFIGURATION
# ------------------------------------------------------------

with open(CONFIG_FILE, "r") as f:
    saved_config = json.load(f)

assert saved_config["fold"] == 4
assert saved_config["seed"] == 2026

assert saved_config["training"]["batch_size"] == 64
assert saved_config["training"]["max_epochs"] == 30
assert saved_config["training"]["learning_rate"] == 1e-4
assert saved_config["training"]["weight_decay"] == 1e-4

print("\n✓ Saved Fold-4 configuration verified")

# ------------------------------------------------------------
# 7. MODEL IMPORT
# ------------------------------------------------------------

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

from neurotrustformer_eeg_model import NeuroTrustFormerEEG

print("✓ NeuroTrustFormerEEG imported")

# ------------------------------------------------------------
# 8. NORMALIZATION
# ------------------------------------------------------------

norm = np.load(NORM_FILE)

NORM_MEANS = norm["means"].astype(np.float32)
NORM_STDS = norm["stds"].astype(np.float32)

assert NORM_MEANS.shape == (17,)
assert NORM_STDS.shape == (17,)

assert np.all(np.isfinite(NORM_MEANS))
assert np.all(np.isfinite(NORM_STDS))
assert np.all(NORM_STDS > 0)

print("✓ 17-channel training-only normalization loaded")

# ------------------------------------------------------------
# 9. TARGET CHANNELS
# ------------------------------------------------------------

TARGET_CHANNELS = [
    "C3-P3",
    "C4-P4",
    "CZ-PZ",
    "F3-C3",
    "F4-C4",
    "F7-T7",
    "F8-T8",
    "FP1-F3",
    "FP1-F7",
    "FP2-F4",
    "FP2-F8",
    "FZ-CZ",
    "P3-O1",
    "P4-O2",
    "P7-O1",
    "P8-O2",
    "T7-P7"
]

assert len(TARGET_CHANNELS) == 17

# ------------------------------------------------------------
# 10. LOAD TRAINING METADATA
# ------------------------------------------------------------

train_meta = pd.read_csv(
    TRAIN_META_FILE
)

train_meta["case"] = (
    train_meta["case"]
    .astype(str)
    .str.lower()
)

train_meta["recording"] = (
    train_meta["recording"]
    .astype(str)
)

train_meta["label"] = (
    train_meta["label"]
    .astype(int)
)

assert set(
    train_meta["case"].unique()
) == set(TRAIN_CASES)

assert train_meta["label"].isin([0, 1]).all()

# ------------------------------------------------------------
# 11. LOAD VALIDATION METADATA
# ------------------------------------------------------------

master_meta = pd.read_csv(
    MASTER_META_FILE
)

master_meta["case"] = (
    master_meta["case"]
    .astype(str)
    .str.lower()
)

master_meta["recording"] = (
    master_meta["recording"]
    .astype(str)
)

master_meta["label"] = (
    master_meta["label"]
    .astype(int)
)

val_meta = master_meta[
    master_meta["case"].isin(VAL_CASES)
].copy()

assert set(
    val_meta["case"].unique()
) == set(VAL_CASES)

assert set(
    train_meta["case"].unique()
).isdisjoint(
    set(val_meta["case"].unique())
)

print("\n" + "=" * 70)
print("DATA PARTITIONS")
print("=" * 70)

print("\nTraining cases:")
print(TRAIN_CASES)

print("\nValidation cases:")
print(VAL_CASES)

print("\nTest cases:")
print(TEST_CASES)

print("\nTraining windows:")
print(f"{len(train_meta):,}")

print("Training recordings:")
print(
    train_meta["recording"].nunique()
)

print("\nValidation windows:")
print(f"{len(val_meta):,}")

print("Validation recordings:")
print(
    val_meta["recording"].nunique()
)

print("\nTraining labels:")
print(
    train_meta["label"]
    .value_counts()
    .sort_index()
    .to_dict()
)

print("\nValidation labels:")
print(
    val_meta["label"]
    .value_counts()
    .sort_index()
    .to_dict()
)

# ------------------------------------------------------------
# 12. EDF LOOKUP
# ------------------------------------------------------------

edf_lookup = {
    f.lower(): f
    for f in os.listdir(DATA_DIR)
    if f.lower().endswith(".edf")
}

print(
    "\nEDF files available:",
    len(edf_lookup)
)

# ------------------------------------------------------------
# 13. FAST RECORDING CACHE
# ------------------------------------------------------------

class RecordingCache:

    def __init__(
        self,
        data_dir,
        means,
        stds
    ):

        self.data_dir = data_dir

        self.means = means.reshape(
            -1, 1
        )

        self.stds = stds.reshape(
            -1, 1
        )

        self.current_recording = None
        self.current_data = None
        self.current_mask = None

    def load_recording(
        self,
        recording
    ):

        if recording == self.current_recording:
            return

        filename = recording

        if not filename.lower().endswith(".edf"):
            filename += ".edf"

        key = filename.lower()

        if key not in edf_lookup:
            raise FileNotFoundError(
                f"EDF not found: {filename}"
            )

        path = os.path.join(
            self.data_dir,
            edf_lookup[key]
        )

        # Duplicate-channel warnings are known for some
        # CHB-MIT EDFs and are handled by explicit mapping.
        with warnings.catch_warnings():
            warnings.filterwarnings(
                "ignore",
                message="Channel names are not unique"
            )

            raw = mne.io.read_raw_edf(
                path,
                preload=True,
                verbose=False
            )

        sfreq = float(
            raw.info["sfreq"]
        )

        if abs(sfreq - 256.0) > 1e-6:

            del raw

            raise ValueError(
                f"{filename}: sampling rate "
                f"{sfreq}, expected 256 Hz"
            )

        # ----------------------------------------------------
        # Channel mapping
        # ----------------------------------------------------

        normalized_names = {}

        for idx, name in enumerate(
            raw.ch_names
        ):

            key_name = (
                name.strip()
                .upper()
            )

            if key_name not in normalized_names:
                normalized_names[key_name] = idx

        channel_indices = []

        for target in TARGET_CHANNELS:

            key_name = (
                target.strip()
                .upper()
            )

            channel_indices.append(
                normalized_names.get(
                    key_name,
                    None
                )
            )

        # ----------------------------------------------------
        # Read each required channel once
        # ----------------------------------------------------

        n_total = raw.n_times

        data = np.zeros(
            (17, n_total),
            dtype=np.float32
        )

        mask = np.zeros(
            17,
            dtype=np.float32
        )

        for channel_idx, source_idx in enumerate(
            channel_indices
        ):

            if source_idx is None:
                continue

            signal = raw.get_data(
                picks=[source_idx],
                verbose=False
            ).astype(
                np.float32
            ).ravel()

            if signal.size != n_total:

                del raw

                raise ValueError(
                    f"Unexpected sample count "
                    f"in {filename}"
                )

            data[channel_idx] = signal
            mask[channel_idx] = 1.0

        del raw

        # ----------------------------------------------------
        # TRAINING-ONLY NORMALIZATION
        # ----------------------------------------------------

        data = (
            data - self.means
        ) / self.stds

        self.current_recording = recording
        self.current_data = data
        self.current_mask = mask

    def get_batch(
        self,
        rows
    ):

        recording = str(
            rows.iloc[0]["recording"]
        )

        self.load_recording(
            recording
        )

        eeg_list = []
        mask_list = []
        label_list = []

        for _, row in rows.iterrows():

            start_sample = int(
                round(
                    float(
                        row["start_sec"]
                    ) * 256
                )
            )

            stop_sample = (
                start_sample + N_SAMPLES
            )

            if (
                start_sample < 0
                or stop_sample >
                self.current_data.shape[1]
            ):
                continue

            eeg = self.current_data[
                :,
                start_sample:stop_sample
            ]

            if eeg.shape != (
                17,
                2560
            ):
                continue

            eeg_list.append(eeg)

            mask_list.append(
                self.current_mask.copy()
            )

            label_list.append(
                int(row["label"])
            )

        if not eeg_list:
            return None

        return (
            np.stack(eeg_list),
            np.stack(mask_list),
            np.asarray(
                label_list,
                dtype=np.int64
            )
        )

# ------------------------------------------------------------
# 14. BATCH GENERATOR
# ------------------------------------------------------------

def generate_batches(
    dataframe,
    cache,
    batch_size,
    shuffle=False
):

    recordings = list(
        dataframe[
            "recording"
        ].astype(str).unique()
    )

    if shuffle:
        random.shuffle(
            recordings
        )

    for recording in recordings:

        rec_df = dataframe[
            dataframe["recording"].astype(str)
            == recording
        ].copy()

        if shuffle:

            rec_df = rec_df.sample(
                frac=1.0,
                random_state=random.randint(
                    0,
                    2_147_483_647
                )
            )

        for start in range(
            0,
            len(rec_df),
            batch_size
        ):

            rows = rec_df.iloc[
                start:start + batch_size
            ]

            result = cache.get_batch(
                rows
            )

            if result is None:
                continue

            yield result

# ------------------------------------------------------------
# 15. CLASS WEIGHTS
# ------------------------------------------------------------

class_counts = (
    train_meta["label"]
    .value_counts()
    .reindex([0, 1])
    .fillna(0)
    .values
)

class_weights_np = (
    class_counts.sum()
    /
    (2.0 * class_counts)
)

class_weights = torch.tensor(
    class_weights_np,
    dtype=torch.float32,
    device=DEVICE
)

print("\n" + "=" * 70)
print("CLASS WEIGHTS")
print("=" * 70)

print(
    "Normal :",
    float(class_weights[0])
)

print(
    "Seizure:",
    float(class_weights[1])
)

# ------------------------------------------------------------
# 16. MODEL
# ------------------------------------------------------------

model = NeuroTrustFormerEEG(
    n_channels=N_CHANNELS,
    n_samples=N_SAMPLES,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    dropout=DROPOUT
).to(DEVICE)

parameter_count = sum(
    p.numel()
    for p in model.parameters()
)

assert parameter_count == 561604

print("\nModel parameters:")
print(
    f"{parameter_count:,}"
)

# ------------------------------------------------------------
# 17. LOSS / OPTIMIZER / SCHEDULER
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=SCHEDULER_FACTOR,
    patience=SCHEDULER_PATIENCE
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True
)

# ------------------------------------------------------------
# 18. TRAINING FUNCTION
# ------------------------------------------------------------

def train_one_epoch():

    model.train()

    total_loss = 0.0
    total_n = 0

    y_true = []
    y_prob = []

    for eeg, masks, labels in generate_batches(
        train_meta,
        train_cache,
        BATCH_SIZE,
        shuffle=True
    ):

        eeg = torch.from_numpy(
            eeg
        ).to(
            DEVICE,
            non_blocking=True
        )

        masks = torch.from_numpy(
            masks
        ).to(
            DEVICE,
            non_blocking=True
        )

        labels_t = torch.from_numpy(
            labels
        ).to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            output = model(
                eeg,
                channel_mask=masks
            )

            logits = output["logits"]

            loss = criterion(
                logits,
                labels_t
            )

        scaler.scale(
            loss
        ).backward()

        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRADIENT_CLIP
        )

        scaler.step(
            optimizer
        )

        scaler.update()

        probabilities = torch.softmax(
            logits,
            dim=1
        )[:, 1]

        n = labels_t.size(0)

        total_loss += (
            loss.item() * n
        )

        total_n += n

        y_true.extend(
            labels.tolist()
        )

        y_prob.extend(
            probabilities
            .detach()
            .float()
            .cpu()
            .numpy()
            .tolist()
        )

    y_true = np.asarray(
        y_true
    )

    y_prob = np.asarray(
        y_prob
    )

    y_pred = (
        y_prob >= 0.5
    ).astype(int)

    metrics = {
        "loss": (
            total_loss / total_n
        ),
        "accuracy": accuracy_score(
            y_true,
            y_pred
        ),
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_true,
            y_prob
        ),
        "pr_auc": average_precision_score(
            y_true,
            y_prob
        )
    }

    return metrics

# ------------------------------------------------------------
# 19. VALIDATION FUNCTION
# ------------------------------------------------------------

@torch.no_grad()
def validate():

    model.eval()

    total_loss = 0.0
    total_n = 0

    y_true = []
    y_prob = []

    for eeg, masks, labels in generate_batches(
        val_meta,
        val_cache,
        BATCH_SIZE,
        shuffle=False
    ):

        eeg = torch.from_numpy(
            eeg
        ).to(
            DEVICE,
            non_blocking=True
        )

        masks = torch.from_numpy(
            masks
        ).to(
            DEVICE,
            non_blocking=True
        )

        labels_t = torch.from_numpy(
            labels
        ).to(
            DEVICE,
            non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            output = model(
                eeg,
                channel_mask=masks
            )

            logits = output["logits"]

            loss = criterion(
                logits,
                labels_t
            )

        probabilities = torch.softmax(
            logits,
            dim=1
        )[:, 1]

        n = labels_t.size(0)

        total_loss += (
            loss.item() * n
        )

        total_n += n

        y_true.extend(
            labels.tolist()
        )

        y_prob.extend(
            probabilities
            .float()
            .cpu()
            .numpy()
            .tolist()
        )

    y_true = np.asarray(
        y_true
    )

    y_prob = np.asarray(
        y_prob
    )

    y_pred = (
        y_prob >= 0.5
    ).astype(int)

    metrics = {
        "loss": (
            total_loss / total_n
        ),
        "accuracy": accuracy_score(
            y_true,
            y_pred
        ),
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_true,
            y_prob
        ),
        "pr_auc": average_precision_score(
            y_true,
            y_prob
        )
    }

    return metrics

# ------------------------------------------------------------
# 20. RECORDING CACHES
# ------------------------------------------------------------

train_cache = RecordingCache(
    DATA_DIR,
    NORM_MEANS,
    NORM_STDS
)

val_cache = RecordingCache(
    DATA_DIR,
    NORM_MEANS,
    NORM_STDS
)

# ------------------------------------------------------------
# 21. TRAINING LOOP
# ------------------------------------------------------------

history = []

best_val_loss = float("inf")
best_epoch = None
epochs_without_improvement = 0

print("\n" + "=" * 70)
print("STARTING FOLD-4 GPU TRAINING")
print("=" * 70)

print(
    f"\nMaximum epochs: {MAX_EPOCHS}"
)

print(
    f"Batch size: {BATCH_SIZE}"
)

print(
    f"Learning rate: {LEARNING_RATE}"
)

print(
    f"Early stopping patience: "
    f"{EARLY_STOPPING_PATIENCE}"
)

print(
    "\nIMPORTANT: Fold-4 test data are NOT loaded "
    "or evaluated during training."
)

training_start = time.time()

for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    epoch_start = time.time()

    train_metrics = train_one_epoch()

    val_metrics = validate()

    scheduler.step(
        val_metrics["loss"]
    )

    current_lr = (
        optimizer.param_groups[0]["lr"]
    )

    epoch_time = (
        time.time()
        - epoch_start
    )

    row = {
        "epoch": epoch,

        "train_loss":
            train_metrics["loss"],

        "train_accuracy":
            train_metrics["accuracy"],

        "train_precision":
            train_metrics["precision"],

        "train_recall":
            train_metrics["recall"],

        "train_f1":
            train_metrics["f1"],

        "train_roc_auc":
            train_metrics["roc_auc"],

        "train_pr_auc":
            train_metrics["pr_auc"],

        "val_loss":
            val_metrics["loss"],

        "val_accuracy":
            val_metrics["accuracy"],

        "val_precision":
            val_metrics["precision"],

        "val_recall":
            val_metrics["recall"],

        "val_f1":
            val_metrics["f1"],

        "val_roc_auc":
            val_metrics["roc_auc"],

        "val_pr_auc":
            val_metrics["pr_auc"],

        "learning_rate":
            current_lr,

        "epoch_time_sec":
            epoch_time
    }

    history.append(row)

    print(
        f"\nEpoch {epoch:02d}/{MAX_EPOCHS} "
        f"| {epoch_time:.1f}s"
    )

    print(
        f"Train | "
        f"loss={train_metrics['loss']:.5f} "
        f"acc={train_metrics['accuracy']:.4f} "
        f"F1={train_metrics['f1']:.4f} "
        f"AUC={train_metrics['roc_auc']:.4f} "
        f"PR-AUC={train_metrics['pr_auc']:.4f}"
    )

    print(
        f"Val   | "
        f"loss={val_metrics['loss']:.5f} "
        f"acc={val_metrics['accuracy']:.4f} "
        f"F1={val_metrics['f1']:.4f} "
        f"AUC={val_metrics['roc_auc']:.4f} "
        f"PR-AUC={val_metrics['pr_auc']:.4f}"
    )

    print(
        f"LR={current_lr:.2e}"
    )

    # --------------------------------------------------------
    # BEST MODEL
    # --------------------------------------------------------

    if (
        val_metrics["loss"]
        < best_val_loss
    ):

        best_val_loss = (
            val_metrics["loss"]
        )

        best_epoch = epoch

        epochs_without_improvement = 0

        checkpoint = {
            "epoch": epoch,
            "model_state_dict":
                model.state_dict(),
            "optimizer_state_dict":
                optimizer.state_dict(),
            "scheduler_state_dict":
                scheduler.state_dict(),
            "best_val_loss":
                best_val_loss,
            "config":
                saved_config,
            "seed":
                SEED,
            "fold":
                FOLD
        }

        torch.save(
            checkpoint,
            BEST_MODEL_FILE
        )

        print(
            f"✓ BEST CHECKPOINT SAVED "
            f"(epoch {epoch}, "
            f"val_loss={best_val_loss:.5f})"
        )

    else:

        epochs_without_improvement += 1

        print(
            f"No validation improvement: "
            f"{epochs_without_improvement}/"
            f"{EARLY_STOPPING_PATIENCE}"
        )

    # --------------------------------------------------------
    # SAVE HISTORY
    # --------------------------------------------------------

    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_FILE,
        index=False
    )

    # --------------------------------------------------------
    # EARLY STOPPING
    # --------------------------------------------------------

    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):

        print(
            "\nEarly stopping triggered."
        )

        break

# ------------------------------------------------------------
# 22. FINAL SUMMARY
# ------------------------------------------------------------

total_time = (
    time.time()
    - training_start
)

history_df = pd.DataFrame(
    history
)

history_df.to_csv(
    HISTORY_FILE,
    index=False
)

print("\n" + "=" * 70)
print("FOLD-4 TRAINING COMPLETE")
print("=" * 70)

print(
    "Epochs completed:",
    len(history_df)
)

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best validation loss:",
    f"{best_val_loss:.6f}"
)

print(
    "Total training time:",
    f"{total_time / 60:.2f} minutes"
)

print(
    "\nBest checkpoint:"
)

print(
    BEST_MODEL_FILE
)

print(
    "\nTraining history:"
)

print(
    HISTORY_FILE
)

print("\nTEST SET STATUS:")
print(
    "✓ Fold-4 test data were NOT loaded."
)

print(
    "✓ Test threshold was NOT tuned."
)

print(
    "✓ Model selection used validation loss only."
)

print("\n" + "=" * 70)
print(
    "✓ FOLD-4 TRAINING FINISHED"
)
print(
    "✓ TEST SET REMAINS COMPLETELY UNTOUCHED"
)
print("=" * 70)

In [ ]:
# ============================================================
# FOLD-5 ACTUAL MODEL TRAINING
# ============================================================
# NeuroTrustFormer EEG
#
# Fold-5:
#   Train       : 16 cases
#   Validation  : 4 cases
#   Test        : 4 cases  <-- NOT TOUCHED HERE
#
# IMPORTANT:
#   - Training uses Fold-5 TRAINING cases only
#   - Validation is used for model selection/early stopping
#   - TEST data are NOT accessed
#   - Training-only normalization is used
#   - Best model is selected by validation loss
# ============================================================

import os
import re
import json
import time
import random
import numpy as np
import pandas as pd
import mne
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

BASE = "/content/drive/MyDrive/NeuroTrustFormer_HISS"

RESULTS_DIR = f"{BASE}/results"
DATA_DIR = f"{BASE}/data/chbmit"
CODE_DIR = f"{BASE}/code"
MODELS_DIR = f"{BASE}/models"

TRAIN_METADATA_FILE = (
    f"{RESULTS_DIR}/"
    "CHBMIT_FOLD5_training_metadata.csv"
)

NORMALIZATION_FILE = (
    f"{RESULTS_DIR}/"
    "CHBMIT_FOLD5_training_normalization_parameters.npz"
)

SPLIT_FILE = (
    f"{RESULTS_DIR}/"
    "CHBMIT_locked_case_level_5fold_split.csv"
)

CONFIG_FILE = (
    f"{RESULTS_DIR}/"
    "CHBMIT_FOLD5_training_config.json"
)

HISTORY_FILE = (
    f"{RESULTS_DIR}/"
    "CHBMIT_FOLD5_training_history.csv"
)

BEST_CHECKPOINT = (
    f"{MODELS_DIR}/"
    "NeuroTrustFormer_FOLD5_best.pt"
)

# ------------------------------------------------------------
# REPRODUCIBILITY
# ------------------------------------------------------------

SEED = 2026

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# TRAINING CONFIGURATION
# ------------------------------------------------------------

FOLD = 5

BATCH_SIZE = 64
MAX_EPOCHS = 30

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

EARLY_STOPPING_PATIENCE = 5

GRADIENT_CLIP = 1.0

# Model
N_CHANNELS = 17
N_SAMPLES = 2560

D_MODEL = 128
N_HEADS = 4
N_LAYERS = 2
DROPOUT = 0.20

# ------------------------------------------------------------
# TARGET CHANNELS
# ------------------------------------------------------------

TARGET_CHANNELS = [
    "C3-P3",
    "C4-P4",
    "CZ-PZ",
    "F3-C3",
    "F4-C4",
    "F7-T7",
    "F8-T8",
    "FP1-F3",
    "FP1-F7",
    "FP2-F4",
    "FP2-F8",
    "FZ-CZ",
    "P3-O1",
    "P4-O2",
    "P7-O1",
    "P8-O2",
    "T7-P7",
]

# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 70)
print("FOLD-5 ACTUAL TRAINING")
print("=" * 70)

print("\nDevice:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

# ------------------------------------------------------------
# FILE CHECKS
# ------------------------------------------------------------

assert os.path.exists(
    TRAIN_METADATA_FILE
)

assert os.path.exists(
    NORMALIZATION_FILE
)

assert os.path.exists(
    SPLIT_FILE
)

assert os.path.exists(
    f"{CODE_DIR}/neurotrustformer_eeg_model.py"
)

os.makedirs(
    MODELS_DIR,
    exist_ok=True
)

# ------------------------------------------------------------
# LOAD SPLIT
# ------------------------------------------------------------

split = pd.read_csv(
    SPLIT_FILE
)

split["case"] = (
    split["case"]
    .astype(str)
    .str.lower()
)

TRAIN_CASES = sorted(
    split.loc[
        (split["fold"] == FOLD)
        & (split["role"] == "train"),
        "case"
    ].unique()
)

VAL_CASES = sorted(
    split.loc[
        (split["fold"] == FOLD)
        & (split["role"] == "validation"),
        "case"
    ].unique()
)

TEST_CASES = sorted(
    split.loc[
        (split["fold"] == FOLD)
        & (split["role"] == "test"),
        "case"
    ].unique()
)

# ------------------------------------------------------------
# CRITICAL SPLIT CHECK
# ------------------------------------------------------------

assert len(TRAIN_CASES) == 16
assert len(VAL_CASES) == 4
assert len(TEST_CASES) == 4

assert set(TRAIN_CASES).isdisjoint(
    set(VAL_CASES)
)

assert set(TRAIN_CASES).isdisjoint(
    set(TEST_CASES)
)

assert set(VAL_CASES).isdisjoint(
    set(TEST_CASES)
)

print("\nFold-5 split:")
print(
    "Training cases   :",
    len(TRAIN_CASES)
)

print(
    "Validation cases :",
    len(VAL_CASES)
)

print(
    "Test cases       :",
    len(TEST_CASES)
)

print(
    "\nTest cases will NOT be loaded during training."
)

# ------------------------------------------------------------
# LOAD METADATA
# ------------------------------------------------------------

metadata = pd.read_csv(
    TRAIN_METADATA_FILE
)

metadata["case"] = (
    metadata["case"]
    .astype(str)
    .str.lower()
)

assert set(
    metadata["case"].unique()
).issubset(
    set(TRAIN_CASES)
)

print("\nTraining metadata rows:")
print(
    f"{len(metadata):,}"
)

print(
    "Training recordings:",
    metadata["recording"].nunique()
)

# ------------------------------------------------------------
# CONVERT LABEL
# ------------------------------------------------------------

metadata["label"] = (
    metadata["label"]
    .astype(int)
)

assert set(
    metadata["label"].unique()
) == {0, 1}

# ------------------------------------------------------------
# CREATE VALIDATION METADATA
# ------------------------------------------------------------
# Validation metadata is selected from the MASTER metadata.
# Test metadata is deliberately never loaded.

MASTER_METADATA_FILE = (
    f"{RESULTS_DIR}/"
    "CHBMIT_MASTER_24CASE_window_metadata.csv"
)

assert os.path.exists(
    MASTER_METADATA_FILE
)

master = pd.read_csv(
    MASTER_METADATA_FILE
)

master["case"] = (
    master["case"]
    .astype(str)
    .str.lower()
)

master["label"] = (
    master["label"]
    .astype(int)
)

val_metadata = master[
    master["case"].isin(
        VAL_CASES
    )
].copy()

assert len(val_metadata) > 0

assert set(
    val_metadata["case"].unique()
).issubset(
    set(VAL_CASES)
)

print(
    "Validation metadata rows:",
    f"{len(val_metadata):,}"
)

print(
    "Validation recordings:",
    val_metadata["recording"].nunique()
)

# ------------------------------------------------------------
# VERIFY TEST CASES ARE NOT PRESENT
# ------------------------------------------------------------

assert not set(
    metadata["case"].unique()
).intersection(
    set(TEST_CASES)
)

assert not set(
    val_metadata["case"].unique()
).intersection(
    set(TEST_CASES)
)

print(
    "Train/validation vs test case separation: PASS"
)

# ------------------------------------------------------------
# LOAD NORMALIZATION PARAMETERS
# ------------------------------------------------------------

normalization = np.load(
    NORMALIZATION_FILE,
    allow_pickle=False
)

NORM_MEAN = normalization[
    "mean"
].astype(
    np.float32
)

NORM_STD = normalization[
    "std"
].astype(
    np.float32
)

assert NORM_MEAN.shape == (
    N_CHANNELS,
)

assert NORM_STD.shape == (
    N_CHANNELS,
)

assert np.all(
    np.isfinite(NORM_MEAN)
)

assert np.all(
    np.isfinite(NORM_STD)
)

assert np.all(
    NORM_STD > 0
)

print(
    "Training-only normalization loaded: PASS"
)

# ------------------------------------------------------------
# CHANNEL NAME NORMALIZATION
# ------------------------------------------------------------

def normalize_channel_name(name):

    name = str(
        name
    ).upper().strip()

    name = name.replace(
        "EEG ",
        ""
    )

    name = name.replace(
        "-REF",
        ""
    )

    name = name.replace(
        " ",
        ""
    )

    name = name.replace(
        ".",
        ""
    )

    name = name.replace(
        "_",
        "-"
    )

    name = name.replace(
        "--",
        "-"
    )

    return name

# ------------------------------------------------------------
# EDF RECORDING CACHE
# ------------------------------------------------------------

def load_recording_windows(
    edf_path,
    recording_rows
):
    """
    Load one EDF once and extract all requested windows.

    Returns:
        eeg       : [N, 17, 2560]
        masks     : [N, 17]
        labels    : [N]
        row_order : original dataframe indices
    """

    raw = mne.io.read_raw_edf(
        edf_path,
        preload=True,
        verbose="ERROR"
    )

    sfreq = float(
        raw.info["sfreq"]
    )

    assert np.isclose(
        sfreq,
        256.0
    ), (
        f"Unexpected sampling rate "
        f"{sfreq} in {edf_path}"
    )

    # --------------------------------------------------------
    # Build channel map
    # --------------------------------------------------------

    channel_map = {}

    for idx, ch_name in enumerate(
        raw.ch_names
    ):

        normalized = normalize_channel_name(
            ch_name
        )

        if normalized not in channel_map:
            channel_map[normalized] = idx

    # --------------------------------------------------------
    # Load target channels once
    # --------------------------------------------------------

    channel_data = []

    channel_available = []

    for target in TARGET_CHANNELS:

        target_norm = normalize_channel_name(
            target
        )

        source_idx = channel_map.get(
            target_norm
        )

        if source_idx is None:

            channel_data.append(
                None
            )

            channel_available.append(
                False
            )

        else:

            data = raw.get_data(
                picks=[source_idx]
            )[0]

            data = np.asarray(
                data,
                dtype=np.float32
            )

            channel_data.append(
                data
            )

            channel_available.append(
                True
            )

    # --------------------------------------------------------
    # Extract windows
    # --------------------------------------------------------

    n_rows = len(
        recording_rows
    )

    eeg = np.zeros(
        (
            n_rows,
            N_CHANNELS,
            N_SAMPLES
        ),
        dtype=np.float32
    )

    masks = np.zeros(
        (
            n_rows,
            N_CHANNELS
        ),
        dtype=np.float32
    )

    labels = (
        recording_rows["label"]
        .astype(int)
        .to_numpy()
    )

    # Preserve metadata row order
    row_order = (
        recording_rows.index.to_numpy()
    )

    # --------------------------------------------------------
    # Determine window start positions
    # --------------------------------------------------------

    if "start_sec" in recording_rows.columns:

        start_seconds = (
            recording_rows[
                "start_sec"
            ].astype(float).to_numpy()
        )

    elif "window_start_sec" in recording_rows.columns:

        start_seconds = (
            recording_rows[
                "window_start_sec"
            ].astype(float).to_numpy()
        )

    else:

        raise KeyError(
            "Could not find start_sec or "
            "window_start_sec in metadata."
        )

    for i, start_sec in enumerate(
        start_seconds
    ):

        start_sample = int(
            round(
                float(start_sec)
                * 256.0
            )
        )

        stop_sample = (
            start_sample
            + N_SAMPLES
        )

        for ch_idx in range(
            N_CHANNELS
        ):

            data = channel_data[
                ch_idx
            ]

            if data is None:
                continue

            if (
                start_sample < 0
                or stop_sample > len(data)
            ):
                continue

            segment = data[
                start_sample:stop_sample
            ]

            if len(segment) != N_SAMPLES:
                continue

            # Training-only z-score normalization
            segment = (
                segment
                - NORM_MEAN[ch_idx]
            ) / NORM_STD[ch_idx]

            eeg[
                i,
                ch_idx,
                :
            ] = segment

            masks[
                i,
                ch_idx
            ] = 1.0

    del raw

    return (
        eeg,
        masks,
        labels,
        row_order
    )

# ------------------------------------------------------------
# GROUP METADATA BY RECORDING
# ------------------------------------------------------------

train_groups = [
    group
    for _, group
    in metadata.groupby(
        "recording",
        sort=True
    )
]

val_groups = [
    group
    for _, group
    in val_metadata.groupby(
        "recording",
        sort=True
    )
]

print(
    "\nTraining recording groups:",
    len(train_groups)
)

print(
    "Validation recording groups:",
    len(val_groups)
)

# ------------------------------------------------------------
# MODEL IMPORT
# ------------------------------------------------------------

import sys

if CODE_DIR not in sys.path:
    sys.path.insert(
        0,
        CODE_DIR
    )

from neurotrustformer_eeg_model import (
    NeuroTrustFormerEEG
)

# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------

model = NeuroTrustFormerEEG(
    n_channels=N_CHANNELS,
    n_samples=N_SAMPLES,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    dropout=DROPOUT
).to(device)

total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

print(
    "\nModel parameters:",
    f"{total_parameters:,}"
)

# ------------------------------------------------------------
# TRAINING CLASS WEIGHTS
# ------------------------------------------------------------

normal_count = int(
    (metadata["label"] == 0).sum()
)

seizure_count = int(
    (metadata["label"] == 1).sum()
)

total_count = (
    normal_count
    + seizure_count
)

class_weights = torch.tensor(
    [
        total_count
        / (2.0 * normal_count),

        total_count
        / (2.0 * seizure_count)
    ],
    dtype=torch.float32,
    device=device
)

print(
    "\nClass weights:",
    class_weights.detach().cpu().numpy()
)

# ------------------------------------------------------------
# LOSS / OPTIMIZER / SCHEDULER
# ------------------------------------------------------------

criterion = torch.nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

# ------------------------------------------------------------
# MIXED PRECISION
# ------------------------------------------------------------

use_amp = torch.cuda.is_available()

if use_amp:

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=True
    )

else:

    scaler = None

print(
    "\nMixed precision:",
    use_amp
)

# ------------------------------------------------------------
# METRIC FUNCTION
# ------------------------------------------------------------

def calculate_metrics(
    y_true,
    y_prob,
    threshold=0.5
):

    y_true = np.asarray(
        y_true
    )

    y_prob = np.asarray(
        y_prob
    )

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    metrics = {}

    metrics["accuracy"] = accuracy_score(
        y_true,
        y_pred
    )

    metrics["precision"] = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    metrics["recall"] = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    metrics["specificity"] = (
        (
            (y_true == 0)
            & (y_pred == 0)
        ).sum()
        /
        max(
            (y_true == 0).sum(),
            1
        )
    )

    metrics["f1"] = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    if len(
        np.unique(y_true)
    ) == 2:

        metrics["roc_auc"] = roc_auc_score(
            y_true,
            y_prob
        )

        metrics["pr_auc"] = average_precision_score(
            y_true,
            y_prob
        )

    else:

        metrics["roc_auc"] = np.nan
        metrics["pr_auc"] = np.nan

    return metrics

# ------------------------------------------------------------
# TRAINING EPOCH FUNCTION
# ------------------------------------------------------------

def train_one_epoch(
    model,
    groups
):

    model.train()

    epoch_loss = 0.0
    total_samples = 0

    all_labels = []
    all_probs = []

    start_time = time.time()

    # Shuffle recordings, not individual windows.
    shuffled_groups = groups.copy()

    random.shuffle(
        shuffled_groups
    )

    for group_idx, group in enumerate(
        shuffled_groups,
        start=1
    ):

        recording = str(
            group["recording"].iloc[0]
        )

        filename = recording

        if not filename.lower().endswith(
            ".edf"
        ):
            filename += ".edf"

        edf_path = os.path.join(
            DATA_DIR,
            filename
        )

        assert os.path.exists(
            edf_path
        ), (
            f"Training EDF missing: "
            f"{edf_path}"
        )

        eeg_np, mask_np, labels_np, _ = (
            load_recording_windows(
                edf_path,
                group
            )
        )

        n_windows = len(
            labels_np
        )

        # Shuffle windows within recording
        order = np.random.permutation(
            n_windows
        )

        for start in range(
            0,
            n_windows,
            BATCH_SIZE
        ):

            batch_idx = order[
                start:
                start + BATCH_SIZE
            ]

            batch_eeg = torch.from_numpy(
                eeg_np[batch_idx]
            ).to(
                device,
                non_blocking=True
            )

            batch_mask = torch.from_numpy(
                mask_np[batch_idx]
            ).to(
                device,
                non_blocking=True
            )

            batch_labels = torch.from_numpy(
                labels_np[batch_idx]
            ).long().to(
                device,
                non_blocking=True
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            if use_amp:

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16
                ):

                    outputs = model(
                        batch_eeg,
                        channel_mask=batch_mask
                    )

                    loss = criterion(
                        outputs["logits"],
                        batch_labels
                    )

                scaler.scale(
                    loss
                ).backward()

                scaler.unscale_(
                    optimizer
                )

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    GRADIENT_CLIP
                )

                scaler.step(
                    optimizer
                )

                scaler.update()

            else:

                outputs = model(
                    batch_eeg,
                    channel_mask=batch_mask
                )

                loss = criterion(
                    outputs["logits"],
                    batch_labels
                )

                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    GRADIENT_CLIP
                )

                optimizer.step()

            batch_size_actual = len(
                batch_labels
            )

            epoch_loss += (
                float(
                    loss.detach().cpu()
                )
                * batch_size_actual
            )

            total_samples += (
                batch_size_actual
            )

            probs = torch.softmax(
                outputs["logits"],
                dim=1
            )[:, 1]

            all_labels.extend(
                batch_labels.detach()
                .cpu()
                .numpy()
                .tolist()
            )

            all_probs.extend(
                probs.detach()
                .cpu()
                .numpy()
                .tolist()
            )

        if (
            group_idx == 1
            or group_idx % 10 == 0
            or group_idx == len(
                shuffled_groups
            )
        ):

            elapsed = (
                time.time()
                - start_time
            )

            print(
                f"  Train recording "
                f"{group_idx:03d}/"
                f"{len(shuffled_groups):03d} | "
                f"{recording} | "
                f"elapsed {elapsed/60:.1f} min"
            )

    epoch_loss /= max(
        total_samples,
        1
    )

    metrics = calculate_metrics(
        all_labels,
        all_probs
    )

    return (
        epoch_loss,
        metrics
    )

# ------------------------------------------------------------
# VALIDATION FUNCTION
# ------------------------------------------------------------

@torch.no_grad()
def validate_one_epoch(
    model,
    groups
):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_labels = []
    all_probs = []

    start_time = time.time()

    for group_idx, group in enumerate(
        groups,
        start=1
    ):

        recording = str(
            group["recording"].iloc[0]
        )

        filename = recording

        if not filename.lower().endswith(
            ".edf"
        ):
            filename += ".edf"

        edf_path = os.path.join(
            DATA_DIR,
            filename
        )

        assert os.path.exists(
            edf_path
        ), (
            f"Validation EDF missing: "
            f"{edf_path}"
        )

        eeg_np, mask_np, labels_np, _ = (
            load_recording_windows(
                edf_path,
                group
            )
        )

        n_windows = len(
            labels_np
        )

        # Validation order remains deterministic.
        for start in range(
            0,
            n_windows,
            BATCH_SIZE
        ):

            end = min(
                start + BATCH_SIZE,
                n_windows
            )

            batch_eeg = torch.from_numpy(
                eeg_np[start:end]
            ).to(
                device,
                non_blocking=True
            )

            batch_mask = torch.from_numpy(
                mask_np[start:end]
            ).to(
                device,
                non_blocking=True
            )

            batch_labels = torch.from_numpy(
                labels_np[start:end]
            ).long().to(
                device,
                non_blocking=True
            )

            if use_amp:

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16
                ):

                    outputs = model(
                        batch_eeg,
                        channel_mask=batch_mask
                    )

                    loss = criterion(
                        outputs["logits"],
                        batch_labels
                    )

            else:

                outputs = model(
                    batch_eeg,
                    channel_mask=batch_mask
                )

                loss = criterion(
                    outputs["logits"],
                    batch_labels
                )

            batch_size_actual = len(
                batch_labels
            )

            total_loss += (
                float(
                    loss.detach().cpu()
                )
                * batch_size_actual
            )

            total_samples += (
                batch_size_actual
            )

            probs = torch.softmax(
                outputs["logits"],
                dim=1
            )[:, 1]

            all_labels.extend(
                batch_labels.cpu()
                .numpy()
                .tolist()
            )

            all_probs.extend(
                probs.cpu()
                .numpy()
                .tolist()
            )

        if (
            group_idx == 1
            or group_idx % 10 == 0
            or group_idx == len(groups)
        ):

            elapsed = (
                time.time()
                - start_time
            )

            print(
                f"  Validation recording "
                f"{group_idx:03d}/"
                f"{len(groups):03d} | "
                f"{recording} | "
                f"elapsed {elapsed/60:.1f} min"
            )

    val_loss = total_loss / max(
        total_samples,
        1
    )

    metrics = calculate_metrics(
        all_labels,
        all_probs
    )

    return (
        val_loss,
        metrics
    )

# ------------------------------------------------------------
# TRAINING LOOP
# ------------------------------------------------------------

history = []

best_val_loss = float(
    "inf"
)

best_epoch = None

patience_counter = 0

training_start = time.time()

print("\n" + "=" * 70)
print("STARTING REAL FOLD-5 TRAINING")
print("=" * 70)

print(
    f"\nMaximum epochs: {MAX_EPOCHS}"
)

print(
    f"Batch size: {BATCH_SIZE}"
)

print(
    f"Early stopping patience: "
    f"{EARLY_STOPPING_PATIENCE}"
)

print(
    "\nTEST DATA WILL NOT BE USED."
)

# ------------------------------------------------------------
# EPOCH LOOP
# ------------------------------------------------------------

for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    epoch_start = time.time()

    print(
        "\n" + "=" * 70
    )

    print(
        f"EPOCH {epoch}/{MAX_EPOCHS}"
    )

    print(
        "=" * 70
    )

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    train_loss, train_metrics = (
        train_one_epoch(
            model,
            train_groups
        )
    )

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    print(
        "\nRunning validation..."
    )

    val_loss, val_metrics = (
        validate_one_epoch(
            model,
            val_groups
        )
    )

    # --------------------------------------------------------
    # LR SCHEDULER
    # --------------------------------------------------------

    scheduler.step(
        val_loss
    )

    current_lr = optimizer.param_groups[
        0
    ]["lr"]

    epoch_time = (
        time.time()
        - epoch_start
    )

    # --------------------------------------------------------
    # HISTORY RECORD
    # --------------------------------------------------------

    row = {
        "epoch": epoch,

        "train_loss":
            train_loss,

        "train_accuracy":
            train_metrics["accuracy"],

        "train_precision":
            train_metrics["precision"],

        "train_recall":
            train_metrics["recall"],

        "train_specificity":
            train_metrics["specificity"],

        "train_f1":
            train_metrics["f1"],

        "train_roc_auc":
            train_metrics["roc_auc"],

        "train_pr_auc":
            train_metrics["pr_auc"],

        "val_loss":
            val_loss,

        "val_accuracy":
            val_metrics["accuracy"],

        "val_precision":
            val_metrics["precision"],

        "val_recall":
            val_metrics["recall"],

        "val_specificity":
            val_metrics["specificity"],

        "val_f1":
            val_metrics["f1"],

        "val_roc_auc":
            val_metrics["roc_auc"],

        "val_pr_auc":
            val_metrics["pr_auc"],

        "learning_rate":
            current_lr,

        "epoch_time_seconds":
            epoch_time
    }

    history.append(
        row
    )

    history_df = pd.DataFrame(
        history
    )

    history_df.to_csv(
        HISTORY_FILE,
        index=False
    )

    # --------------------------------------------------------
    # PRINT RESULTS
    # --------------------------------------------------------

    print(
        "\n" + "-" * 70
    )

    print(
        f"Epoch {epoch} results"
    )

    print(
        f"Train loss : "
        f"{train_loss:.6f}"
    )

    print(
        f"Train acc  : "
        f"{train_metrics['accuracy']:.6f}"
    )

    print(
        f"Train F1   : "
        f"{train_metrics['f1']:.6f}"
    )

    print(
        f"Train AUC  : "
        f"{train_metrics['roc_auc']:.6f}"
    )

    print(
        f"Val loss   : "
        f"{val_loss:.6f}"
    )

    print(
        f"Val acc    : "
        f"{val_metrics['accuracy']:.6f}"
    )

    print(
        f"Val precision : "
        f"{val_metrics['precision']:.6f}"
    )

    print(
        f"Val recall : "
        f"{val_metrics['recall']:.6f}"
    )

    print(
        f"Val F1     : "
        f"{val_metrics['f1']:.6f}"
    )

    print(
        f"Val ROC-AUC: "
        f"{val_metrics['roc_auc']:.6f}"
    )

    print(
        f"Val PR-AUC : "
        f"{val_metrics['pr_auc']:.6f}"
    )

    print(
        f"Learning rate: "
        f"{current_lr:.8f}"
    )

    print(
        f"Epoch time: "
        f"{epoch_time/60:.2f} min"
    )

    # --------------------------------------------------------
    # BEST CHECKPOINT
    # --------------------------------------------------------

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0

        checkpoint = {
            "fold": FOLD,
            "epoch": epoch,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "scheduler_state_dict":
                scheduler.state_dict(),

            "best_val_loss":
                best_val_loss,

            "val_metrics":
                val_metrics,

            "config": {
                "seed": SEED,
                "batch_size": BATCH_SIZE,
                "max_epochs": MAX_EPOCHS,
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "early_stopping_patience":
                    EARLY_STOPPING_PATIENCE,
                "gradient_clip":
                    GRADIENT_CLIP,

                "n_channels":
                    N_CHANNELS,

                "n_samples":
                    N_SAMPLES,

                "d_model":
                    D_MODEL,

                "n_heads":
                    N_HEADS,

                "n_layers":
                    N_LAYERS,

                "dropout":
                    DROPOUT
            },

            "train_cases":
                TRAIN_CASES,

            "validation_cases":
                VAL_CASES,

            "test_cases":
                TEST_CASES,

            "normalization_file":
                NORMALIZATION_FILE
        }

        torch.save(
            checkpoint,
            BEST_CHECKPOINT
        )

        print(
            "\n*** NEW BEST CHECKPOINT SAVED ***"
        )

        print(
            "Best epoch:",
            best_epoch
        )

        print(
            "Best validation loss:",
            f"{best_val_loss:.6f}"
        )

        print(
            BEST_CHECKPOINT
        )

    else:

        patience_counter += 1

        print(
            f"\nNo validation-loss improvement."
        )

        print(
            f"Early-stopping counter: "
            f"{patience_counter}/"
            f"{EARLY_STOPPING_PATIENCE}"
        )

    # --------------------------------------------------------
    # EARLY STOPPING
    # --------------------------------------------------------

    if patience_counter >= (
        EARLY_STOPPING_PATIENCE
    ):

        print(
            "\n" + "=" * 70
        )

        print(
            "EARLY STOPPING TRIGGERED"
        )

        print(
            f"No validation-loss improvement "
            f"for {EARLY_STOPPING_PATIENCE} epochs."
        )

        print(
            "=" * 70
        )

        break

# ------------------------------------------------------------
# FINAL TRAINING SUMMARY
# ------------------------------------------------------------

total_training_time = (
    time.time()
    - training_start
)

history_df = pd.DataFrame(
    history
)

history_df.to_csv(
    HISTORY_FILE,
    index=False
)

# ------------------------------------------------------------
# CHECKPOINT EXISTS
# ------------------------------------------------------------

assert os.path.exists(
    BEST_CHECKPOINT
)

assert os.path.getsize(
    BEST_CHECKPOINT
) > 0

print("\n" + "=" * 70)
print("FOLD-5 TRAINING COMPLETE")
print("=" * 70)

print(
    "Epochs completed:",
    len(history_df)
)

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best validation loss:",
    f"{best_val_loss:.6f}"
)

print(
    "Total training time:",
    f"{total_training_time/60:.2f} minutes"
)

print(
    "\nTraining history saved:"
)

print(
    HISTORY_FILE
)

print(
    "\nBest checkpoint saved:"
)

print(
    BEST_CHECKPOINT
)

print(
    "\nIMPORTANT:"
)

print(
    "✓ Training used Fold-5 training cases only."
)

print(
    "✓ Validation was used only for model selection."
)

print(
    "✓ Fold-5 test cases were NOT accessed."
)

print(
    "✓ Test evaluation will be performed in a separate cell."
)

print("=" * 70)